In [ ]:
# import pandas as pd
# import numpy as np
# import os
# from utils import load_multimodal_data

# data_files = "/data/s.dharia-ra/PEARL/final_multi_modal/data/EEG_data/EEG_data_plv"
# subjects = [f.split('_')[0] for f in os.listdir(data_files) if f.endswith('.npz')]
# subjects = set(subjects)


# # valid fMRI subjects:
# fMRI_data_files = "/data/s.dharia-ra/PEARL/final_multi_modal/data/fMRI_data"
# fMRI_subjects = [f.split('_')[0] for f in os.listdir(fMRI_data_files) if f.endswith('.npz')]
# fMRI_subjects = set(fMRI_subjects)

# classes = ("all")

# multimodal_data = load_multimodal_data(classes=classes, verbose=False)
# fmri_data = multimodal_data['fmri']
# fmri_sort_indices = fmri_data['sub_ids'].argsort()
# fmri_data['sub_ids'] = fmri_data['sub_ids'][fmri_sort_indices]
# fmri_data['X'] = fmri_data['X'][fmri_sort_indices]
# fmri_data['y'] = fmri_data['y'][fmri_sort_indices]



In [ ]:
# unique_fmri_subs = set(fmri_data['sub_ids'])
# # add sub- prefix to fmri subjects
# unique_fmri_subs = set(['sub-' + str(s) for s in unique_fmri_subs])
# common_subjects = subjects.intersection(unique_fmri_subs)
# print(f"Number of subjects with both EEG and fMRI data: {len(common_subjects)}")

In [ ]:
# from sklearn.cluster import KMeans
# import pandas as pd
# import numpy as np
# from scipy.stats import ttest_ind, f_oneway, mannwhitneyu, kruskal, shapiro, levene

# # Load your data (assuming you have already loaded it)
# from utils import load_multimodal_data

# # Parameters
# WIN_STEP = 5
# TR = 0.8
# STEP_SEC = WIN_STEP * TR
# K_STATES_LIST = list(range(5, 11))
# SEED = 100

# # Load data for specific classes (example: N vs A+P-)
# classes = ("N", "A+P-")
# multimodal_data = load_multimodal_data(classes=classes, verbose=True)
# fmri_data = multimodal_data['fmri']

# def compute_subject_dwell_times_fmri(X_data, y_data, sub_ids_data, target_subjects, kmeans_model, k_states):
#     subject_features, subject_labels, processed_subjects = [], [], []
    
#     # Get unique subjects only
#     unique_subjects = np.unique(target_subjects)
    
#     for subject in unique_subjects:
#         subject_mask = sub_ids_data == subject
#         if not np.any(subject_mask): 
#             continue
        
#         # Get all time points for this subject
#         subject_fmri = X_data[subject_mask]
#         subject_label = y_data[subject_mask][0]  # All samples for same subject have same label
        
#         # Predict cluster for each time point
#         subject_preds = kmeans_model.predict(subject_fmri)
        
#         # Calculate dwell times (time spent in each state)
#         subject_dwell_times = np.bincount(subject_preds, minlength=k_states) * STEP_SEC
        
#         subject_features.append(subject_dwell_times)
#         subject_labels.append(subject_label)
#         processed_subjects.append(subject)
    
#     return np.array(subject_features), np.array(subject_labels), processed_subjects


# def choose_statistical_test(groups):
#     if len(groups) < 2 or any(g.size == 0 for g in groups):
#         return 'None', 1.0, 0.0
    
#     # Check normality and equal variances
#     try:
#         normality_ok = all(shapiro(g)[1] > 0.05 for g in groups if len(g) >= 3)
#         equal_var_ok = levene(*groups)[1] > 0.05
        
#         if normality_ok and equal_var_ok:
#             # Use parametric tests
#             if len(groups) == 2:
#                 stat, p_val = ttest_ind(*groups)
#                 return 't-test', p_val, stat
#             else:
#                 stat, p_val = f_oneway(*groups)
#                 return 'ANOVA', p_val, stat
#         else:
#             # Use non-parametric tests
#             if len(groups) == 2:
#                 stat, p_val = mannwhitneyu(*groups)
#                 return 'Mann-Whitney', p_val, stat
#             else:
#                 stat, p_val = kruskal(*groups)
#                 return 'Kruskal-Wallis', p_val, stat
#     except:
#         return 'None', 1.0, 0.0

# # Find best dwell-time feature with lowest p-value
# best_overall_pval = 1.0
# best_features = None
# best_labels = None
# best_k_states = None
# best_state_idx = None

# # Update the main loop
# print("Searching for best dwell-time features...")

# for K_STATES in K_STATES_LIST:
#     print(f"Testing K_STATES = {K_STATES}")
    
#     # Fit KMeans
#     kmeans = KMeans(n_clusters=K_STATES, random_state=SEED, n_init="auto", max_iter=300)
#     kmeans.fit(fmri_data['X'])
    
#     # Get unique subjects
#     unique_subjects = np.unique(fmri_data['sub_ids'])
    
#     # Calculate dwell-time features for unique subjects only
#     subject_features, subject_labels, processed_subjects = compute_subject_dwell_times_fmri(
#         X_data=fmri_data['X'],
#         y_data=fmri_data['y'],
#         sub_ids_data=fmri_data['sub_ids'],
#         target_subjects=unique_subjects,  # Pass unique subjects, not all samples
#         kmeans_model=kmeans,
#         k_states=K_STATES
#     )
    
#     print(f"Subject features shape: {subject_features.shape}, Subject labels shape: {subject_labels.shape}")
    
#     # Test each state (dwell-time feature)
#     for state in range(K_STATES):
#         # Create groups for statistical testing
#         groups = [subject_features[subject_labels == label, state] for label in np.unique(subject_labels)]
        
#         # Perform statistical test
#         test_name, p_value, stat = choose_statistical_test(groups)
        
#         print(f"  State {state}: {test_name}, p-value = {p_value:.6f}")
        
#         # Update best feature if this has lower p-value
#         if p_value < best_overall_pval:
#             best_overall_pval = p_value
#             best_features = subject_features
#             best_labels = subject_labels
#             best_k_states = K_STATES
#             best_state_idx = state
#             print(f"    ** New best feature found! K={K_STATES}, State={state}, p={p_value:.6f}")

# print(f"\nBest dwell-time feature:")
# print(f"K_STATES: {best_k_states}")
# print(f"State index: {best_state_idx}")
# print(f"P-value: {best_overall_pval:.6f}")

# # Extract the best single feature
# best_single_feature = best_features[:, best_state_idx]

# print(f"\nBest feature shape: {best_single_feature.shape}")
# print(f"Feature values by group:")
# for label in np.unique(best_labels):
#     group_values = best_single_feature[best_labels == label]
#     print(f"  Group {label}: mean={group_values.mean():.3f}, std={group_values.std():.3f}, n={len(group_values)}")

# import pandas as pd
# import numpy as np

# # valid EEG subjects:
# import os
# EEG_data_files = "/data/s.dharia-ra/PEARL/final_multi_modal/data/EEG_data/EEG_data_plv"
# EEG_subjects = [f.split('_')[0] for f in os.listdir(EEG_data_files) if f.endswith('.npz')]
# EEG_subjects = set(EEG_subjects)
# EEG_subjects

# # valid fMRI subjects:
# fMRI_data_files = "/data/s.dharia-ra/PEARL/final_multi_modal/data/fMRI_data"
# fMRI_subjects = [f.split('_')[0] for f in os.listdir(fMRI_data_files) if f.endswith('.npz')]
# fMRI_subjects = set(fMRI_subjects)

# # common subjects:
# common_subjects = EEG_subjects.intersection(fMRI_subjects)
# print(f"Number of subjects with both EEG and fMRI data: {len(common_subjects)}")

# df = pd.read_csv("./participants.tsv", sep="\t")

# # Create risk group classification
# conditions = [
#     # N group: No APOE e4 and No PICALM GG
#     (~df['APOE_haplotype'].str.contains('e4', case=False)) & (df['PICALM_rs3851179'] != 'G/G'),
    
#     # A+P- group: APOE e4 carriers without PICALM GG
#     (df['APOE_haplotype'].str.contains('e4', case=False)) & (df['PICALM_rs3851179'] != 'G/G'),
    
#     # A+P+ group: APOE e4 carriers with PICALM GG
#     (df['APOE_haplotype'].str.contains('e4', case=False)) & (df['PICALM_rs3851179'] == 'G/G')
# ]

# choices = ['N', 'A+P-', 'A+P+']

# df['Risk_Group'] = np.select(conditions, choices, default='Excluded')

# # Remove APOE e2-only carriers from N group (set them as excluded)
# df.loc[(df['APOE_haplotype'].str.contains('e2', case=False)) & 
#        (~df['APOE_haplotype'].str.contains('e4', case=False)) & 
#        (df['Risk_Group'] == 'N'), 'Risk_Group'] = 'Excluded'

# # Display results
# print("Risk Group Distribution:")
# print(df['Risk_Group'].value_counts())

# print("\nStudy groups only:")
# study_df = df[df['Risk_Group'].isin(['N', 'A+P-', 'A+P+'])].copy()  # Add .copy() to avoid warnings
# print(study_df['Risk_Group'].value_counts())

# # make a group column for genes at the end and give numbers to groups N (0), A+P- (1), A+P+ (2)
# group_mapping = {'N': 0, 'A+P-': 1, 'A+P+': 2}
# study_df['Group'] = study_df['Risk_Group'].map(group_mapping)

# # make another column where EEG_subjects and fMRI_subjects are both present
# study_df['Has_EEG_fMRI'] = study_df['participant_id'].apply(lambda x: x in common_subjects)
# final_df = study_df[study_df['Has_EEG_fMRI']]
# print(f"\nNumber of subjects with both EEG and fMRI data in study groups: {final_df.shape[0]}")

# # Display final group counts
# print("\nFinal group distribution with EEG and fMRI data:")
# print(final_df['Risk_Group'].value_counts())
# print(final_df['Group'].value_counts())



# direction aware dwell-time saved


In [ ]:
from sklearn.cluster import KMeans
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind, f_oneway, mannwhitneyu, kruskal, shapiro, levene
from utils import load_multimodal_data
import os

# Parameters
WIN_STEP = 5
TR = 0.8
STEP_SEC = WIN_STEP * TR
K_STATES_LIST = list(range(5, 11))
SEED = 100
P_VALUE_THRESHOLD = 0.05  # Significance threshold

# Load data
classes = ("all")
multimodal_data = load_multimodal_data(classes=classes, verbose=True)
fmri_data = multimodal_data['fmri']

def compute_subject_dwell_times_fmri(X_data, y_data, sub_ids_data, target_subjects, kmeans_model, k_states):
    subject_features, subject_labels, processed_subjects = [], [], []
    unique_subjects = np.unique(target_subjects)
    
    for subject in unique_subjects:
        subject_mask = sub_ids_data == subject
        if not np.any(subject_mask): 
            continue
        
        subject_fmri = X_data[subject_mask]
        subject_label = y_data[subject_mask][0]
        subject_preds = kmeans_model.predict(subject_fmri)
        subject_dwell_times = np.bincount(subject_preds, minlength=k_states) * STEP_SEC
        
        subject_features.append(subject_dwell_times)
        subject_labels.append(subject_label)
        processed_subjects.append(subject)
    
    return np.array(subject_features), np.array(subject_labels), processed_subjects

def choose_statistical_test(groups):
    if len(groups) < 2 or any(g.size == 0 for g in groups):
        return 'None', 1.0, 0.0
    
    try:
        normality_ok = all(shapiro(g)[1] > 0.05 for g in groups if len(g) >= 3)
        equal_var_ok = levene(*groups)[1] > 0.05
        
        if normality_ok and equal_var_ok:
            if len(groups) == 2:
                stat, p_val = ttest_ind(*groups)
                return 't-test', p_val, stat
            else:
                stat, p_val = f_oneway(*groups)
                return 'ANOVA', p_val, stat
        else:
            if len(groups) == 2:
                stat, p_val = mannwhitneyu(*groups)
                return 'Mann-Whitney', p_val, stat
            else:
                stat, p_val = kruskal(*groups)
                return 'Kruskal-Wallis', p_val, stat
    except:
        return 'None', 1.0, 0.0

def determine_dominant_group(subject_features, subject_labels, state):
    """Determine which risk group has highest mean dwell time for this state"""
    unique_labels = np.unique(subject_labels)
    label_to_group = {0: 'N', 1: 'AposPminus', 2: 'AposPpos'}
    
    group_means = {}
    for label in unique_labels:
        group_dwell_times = subject_features[subject_labels == label, state]
        group_means[label] = np.mean(group_dwell_times)
    
    # Find label with highest mean
    dominant_label = max(group_means, key=group_means.get)
    dominant_group = label_to_group[dominant_label]
    
    return dominant_group, group_means

# Load participant data
EEG_data_files = "/data/s.dharia-ra/PEARL/final_multi_modal/data/EEG_data/EEG_data_plv"
EEG_subjects = set([f.split('_')[0] for f in os.listdir(EEG_data_files) if f.endswith('.npz')])

fMRI_data_files = "/data/s.dharia-ra/PEARL/final_multi_modal/data/fMRI_data"
fMRI_subjects = set([f.split('_')[0] for f in os.listdir(fMRI_data_files) if f.endswith('.npz')])

common_subjects = EEG_subjects.intersection(fMRI_subjects)
print(f"Number of subjects with both EEG and fMRI data: {len(common_subjects)}")

df = pd.read_csv("./participants.tsv", sep="\t")

# Create risk groups
conditions = [
    (~df['APOE_haplotype'].str.contains('e4', case=False)) & (df['PICALM_rs3851179'] != 'G/G'),
    (df['APOE_haplotype'].str.contains('e4', case=False)) & (df['PICALM_rs3851179'] != 'G/G'),
    (df['APOE_haplotype'].str.contains('e4', case=False)) & (df['PICALM_rs3851179'] == 'G/G')
]
choices = ['N', 'A+P-', 'A+P+']
df['Risk_Group'] = np.select(conditions, choices, default='Excluded')

df.loc[(df['APOE_haplotype'].str.contains('e2', case=False)) & 
       (~df['APOE_haplotype'].str.contains('e4', case=False)) & 
       (df['Risk_Group'] == 'N'), 'Risk_Group'] = 'Excluded'

study_df = df[df['Risk_Group'].isin(['N', 'A+P-', 'A+P+'])].copy()
study_df['Has_EEG_fMRI'] = study_df['participant_id'].apply(lambda x: x in common_subjects)
final_df = study_df[study_df['Has_EEG_fMRI']].copy()

print(f"Final dataframe shape: {final_df.shape}")
print(f"Classes distribution: {final_df['Risk_Group'].value_counts()}")

participant_to_idx = {pid: idx for idx, pid in enumerate(final_df['participant_id'])}

# === FIND ALL SIGNIFICANT CLUSTERS (DIRECTION-AWARE) ===
print("\n" + "="*80)
print("FINDING ALL SIGNIFICANT CLUSTERS ACROSS K VALUES (WITH DIRECTION)")
print("="*80)

significant_clusters = []  # Store all significant clusters

for K_STATES in K_STATES_LIST:
    print(f"\nTesting K_STATES = {K_STATES}")
    print("="*60)
    
    # Fit KMeans
    kmeans = KMeans(n_clusters=K_STATES, random_state=SEED, n_init="auto", max_iter=300)
    kmeans.fit(fmri_data['X'])
    
    # Get cluster predictions
    all_cluster_predictions = kmeans.predict(fmri_data['X'])
    unique_subjects = np.unique(fmri_data['sub_ids'])
    
    # Calculate dwell-time features
    subject_features, subject_labels, processed_subjects = compute_subject_dwell_times_fmri(
        X_data=fmri_data['X'],
        y_data=fmri_data['y'],
        sub_ids_data=fmri_data['sub_ids'],
        target_subjects=unique_subjects,
        kmeans_model=kmeans,
        k_states=K_STATES
    )
    
    # Test EACH state (not just the best)
    for state in range(K_STATES):
        groups = [subject_features[subject_labels == label, state] 
                 for label in np.unique(subject_labels)]
        test_name, p_value, stat = choose_statistical_test(groups)
        
        # Check if significant
        is_significant = p_value < P_VALUE_THRESHOLD
        significance_marker = "***" if is_significant else ""
        
        print(f"  State {state}: {test_name}, p={p_value:.6f} {significance_marker}")
        
        # If significant, save this cluster WITH DIRECTION
        if is_significant:
            # Determine which group has highest dwell time
            dominant_group, group_means = determine_dominant_group(
                subject_features, subject_labels, state
            )
            
            # Get windows for this cluster
            cluster_mask = all_cluster_predictions == state
            cluster_windows = fmri_data['X'][cluster_mask]
            
            # Store cluster info
            cluster_info = {
                'K': K_STATES,
                'state': state,
                'p_value': p_value,
                'test': test_name,
                'dominant_group': dominant_group,
                'group_means': group_means,
                'windows': cluster_windows,
                'n_windows': len(cluster_windows),
                'subject_features': subject_features[:, state],
                'processed_subjects': processed_subjects
            }
            
            significant_clusters.append(cluster_info)
            
            print(f"    ✓ SIGNIFICANT! Dominant group: {dominant_group}")
            print(f"      Group means: {group_means}")
            print(f"      Windows: {len(cluster_windows)}")

print(f"\n" + "="*80)
print(f"FOUND {len(significant_clusters)} SIGNIFICANT CLUSTERS")
print("="*80)

# === SAVE ALL SIGNIFICANT CLUSTERS (DIRECTION-AWARE) ===
print("\n" + "="*80)
print("SAVING SIGNIFICANT CLUSTERS WITH DIRECTION-AWARE NAMING")
print("="*80)

output_dir = "fMRI_significant_clusters"
os.makedirs(output_dir, exist_ok=True)

for idx, cluster in enumerate(significant_clusters, 1):
    # Create descriptive filename with dominant group
    filename = (f"cluster_K{cluster['K']}_state{cluster['state']}_"
               f"{cluster['dominant_group']}_higher_"
               f"p{cluster['p_value']:.6f}.npz")
    
    filepath = os.path.join(output_dir, filename)
    
    # Save cluster data
    np.savez_compressed(
        filepath,
        windows=cluster['windows'],
        K=cluster['K'],
        state=cluster['state'],
        p_value=cluster['p_value'],
        test=cluster['test'],
        dominant_group=cluster['dominant_group'],
        group_means=cluster['group_means'],
        n_windows=cluster['n_windows']
    )
    
    print(f"{idx:2d}. {filename}")
    print(f"    Windows: {cluster['n_windows']}, p={cluster['p_value']:.6f}")

# === CREATE DATAFRAME WITH ALL SIGNIFICANT FEATURES ===
print("\n" + "="*80)
print("CREATING DATAFRAME WITH ALL SIGNIFICANT FEATURES")
print("="*80)

for idx, cluster in enumerate(significant_clusters):
    K = cluster['K']
    state = cluster['state']
    dominant_group = cluster['dominant_group']
    p_val = cluster['p_value']
    
    # Create column name with direction
    col_name = f'fMRI_K{K}_state{state}_{dominant_group}_higher'
    final_df[col_name] = np.nan
    
    # Add metadata
    final_df[f'{col_name}_pval'] = p_val
    final_df[f'{col_name}_test'] = cluster['test']
    
    # Fill in values
    for i, subject_id in enumerate(cluster['processed_subjects']):
        participant_id = f'sub-{subject_id}'
        if participant_id in participant_to_idx:
            df_idx = participant_to_idx[participant_id]
            final_df.loc[final_df.index[df_idx], col_name] = cluster['subject_features'][i]

# Add group mapping
group_mapping = {'N': 0, 'A+P-': 1, 'A+P+': 2}
final_df['Group'] = final_df['Risk_Group'].map(group_mapping)

# Save dataframe
output_csv = "fMRI_all_significant_features_direction_aware.csv"
final_df.to_csv(output_csv, index=False)

print(f"\n✓ Saved dataframe: {output_csv}")
print(f"  Shape: {final_df.shape}")
print(f"  Significant feature columns: {len(significant_clusters)}")

# === SUMMARY REPORT ===
print("\n" + "="*80)
print("SUMMARY REPORT")
print("="*80)

print(f"\nTotal significant clusters found: {len(significant_clusters)}")

# Group by dominant group
from collections import Counter
dominant_counts = Counter([c['dominant_group'] for c in significant_clusters])

print(f"\nClusters by dominant group:")
for group, count in dominant_counts.items():
    print(f"  {group} has higher dwell time: {count} clusters")

print(f"\nClusters by K value:")
k_counts = Counter([c['K'] for c in significant_clusters])
for k, count in sorted(k_counts.items()):
    print(f"  K={k}: {count} significant clusters")

print(f"\nP-value range:")
p_values = [c['p_value'] for c in significant_clusters]
print(f"  Min: {min(p_values):.6f}")
print(f"  Max: {max(p_values):.6f}")
print(f"  Mean: {np.mean(p_values):.6f}")

print("\n" + "="*80)
print("✅ COMPLETE! All significant clusters saved with direction-aware naming")
print("="*80)

In [ ]:
# from sklearn.cluster import KMeans
# import pandas as pd
# import numpy as np
# from scipy.stats import ttest_ind, f_oneway, mannwhitneyu, kruskal, shapiro, levene

# # Load your data (assuming you have already loaded it)
# from utils import load_multimodal_data

# # Parameters
# WIN_STEP = 5
# TR = 0.8
# STEP_SEC = WIN_STEP * TR
# K_STATES_LIST = list(range(5, 11))
# SEED = 100

# # Load data for specific classes (example: N vs A+P-)
# classes = ("all")
# multimodal_data = load_multimodal_data(classes=classes, verbose=True)
# fmri_data = multimodal_data['fmri']

# def compute_subject_dwell_times_fmri(X_data, y_data, sub_ids_data, target_subjects, kmeans_model, k_states):
#     subject_features, subject_labels, processed_subjects = [], [], []
    
#     # Get unique subjects only
#     unique_subjects = np.unique(target_subjects)
    
#     for subject in unique_subjects:
#         subject_mask = sub_ids_data == subject
#         if not np.any(subject_mask): 
#             continue
        
#         # Get all time points for this subject
#         subject_fmri = X_data[subject_mask]
#         subject_label = y_data[subject_mask][0]  # All samples for same subject have same label
        
#         # Predict cluster for each time point
#         subject_preds = kmeans_model.predict(subject_fmri)
        
#         # Calculate dwell times (time spent in each state)
#         subject_dwell_times = np.bincount(subject_preds, minlength=k_states) * STEP_SEC
        
#         subject_features.append(subject_dwell_times)
#         subject_labels.append(subject_label)
#         processed_subjects.append(subject)
    
#     return np.array(subject_features), np.array(subject_labels), processed_subjects

# def choose_statistical_test(groups):
#     if len(groups) < 2 or any(g.size == 0 for g in groups):
#         return 'None', 1.0, 0.0
    
#     # Check normality and equal variances
#     try:
#         normality_ok = all(shapiro(g)[1] > 0.05 for g in groups if len(g) >= 3)
#         equal_var_ok = levene(*groups)[1] > 0.05
        
#         if normality_ok and equal_var_ok:
#             # Use parametric tests
#             if len(groups) == 2:
#                 stat, p_val = ttest_ind(*groups)
#                 return 't-test', p_val, stat
#             else:
#                 stat, p_val = f_oneway(*groups)
#                 return 'ANOVA', p_val, stat
#         else:
#             # Use non-parametric tests
#             if len(groups) == 2:
#                 stat, p_val = mannwhitneyu(*groups)
#                 return 'Mann-Whitney', p_val, stat
#             else:
#                 stat, p_val = kruskal(*groups)
#                 return 'Kruskal-Wallis', p_val, stat
#     except:
#         return 'None', 1.0, 0.0

# # Load participant data and create risk groups
# import os
# EEG_data_files = "/data/s.dharia-ra/PEARL/final_multi_modal/data/EEG_data/EEG_data_plv"
# EEG_subjects = [f.split('_')[0] for f in os.listdir(EEG_data_files) if f.endswith('.npz')]
# EEG_subjects = set(EEG_subjects)

# fMRI_data_files = "/data/s.dharia-ra/PEARL/final_multi_modal/data/fMRI_data"
# fMRI_subjects = [f.split('_')[0] for f in os.listdir(fMRI_data_files) if f.endswith('.npz')]
# fMRI_subjects = set(fMRI_subjects)

# common_subjects = EEG_subjects.intersection(fMRI_subjects)
# print(f"Number of subjects with both EEG and fMRI data: {len(common_subjects)}")

# df = pd.read_csv("./participants.tsv", sep="\t")

# # Create risk group classification
# conditions = [
#     (~df['APOE_haplotype'].str.contains('e4', case=False)) & (df['PICALM_rs3851179'] != 'G/G'),
#     (df['APOE_haplotype'].str.contains('e4', case=False)) & (df['PICALM_rs3851179'] != 'G/G'),
#     (df['APOE_haplotype'].str.contains('e4', case=False)) & (df['PICALM_rs3851179'] == 'G/G')
# ]

# choices = ['N', 'A+P-', 'A+P+']

# df['Risk_Group'] = np.select(conditions, choices, default='Excluded')

# # Remove APOE e2-only carriers from N group
# df.loc[(df['APOE_haplotype'].str.contains('e2', case=False)) & 
#        (~df['APOE_haplotype'].str.contains('e4', case=False)) & 
#        (df['Risk_Group'] == 'N'), 'Risk_Group'] = 'Excluded'

# # Filter to only include the specified classes and subjects with both EEG and fMRI data
# if classes == ("all"):
#     study_df = df[df['Risk_Group'].isin(['N', 'A+P-', 'A+P+'])].copy()
# else:
#     study_df = df[df['Risk_Group'].isin(list(classes))].copy()


# study_df['Has_EEG_fMRI'] = study_df['participant_id'].apply(lambda x: x in common_subjects)
# final_df = study_df[study_df['Has_EEG_fMRI']].copy()

# print(f"Final dataframe shape: {final_df.shape}")
# print(f"Classes distribution: {final_df['Risk_Group'].value_counts()}")

# # Create a mapping from participant_id to dataframe index for easy lookup
# participant_to_idx = {pid: idx for idx, pid in enumerate(final_df['participant_id'])}

# # Find best dwell-time feature with lowest p-value and collect significant features for each K
# print("Searching for best dwell-time features...")
# best_overall_pval = 1.0
# best_k_states = None
# best_state_idx = None

# # Dictionary to store the best (most significant) dwell-time feature for each K
# best_dwell_features_per_k = {}

# for K_STATES in K_STATES_LIST:
#     print(f"Testing K_STATES = {K_STATES}")
    
#     # Fit KMeans
#     kmeans = KMeans(n_clusters=K_STATES, random_state=SEED, n_init="auto", max_iter=300)
#     kmeans.fit(fmri_data['X'])
    
#     # Get unique subjects
#     unique_subjects = np.unique(fmri_data['sub_ids'])
    
#     # Calculate dwell-time features for unique subjects only
#     subject_features, subject_labels, processed_subjects = compute_subject_dwell_times_fmri(
#         X_data=fmri_data['X'],
#         y_data=fmri_data['y'],
#         sub_ids_data=fmri_data['sub_ids'],
#         target_subjects=unique_subjects,
#         kmeans_model=kmeans,
#         k_states=K_STATES
#     )
    
#     print(f"Subject features shape: {subject_features.shape}, Subject labels shape: {subject_labels.shape}")
    
#     # Find the best state (lowest p-value) for this K
#     best_pval_for_k = 1.0
#     best_state_for_k = None
    
#     for state in range(K_STATES):
#         groups = [subject_features[subject_labels == label, state] for label in np.unique(subject_labels)]
#         test_name, p_value, stat = choose_statistical_test(groups)
        
#         print(f"  State {state}: {test_name}, p-value = {p_value:.6f}")
        
#         # Update best for this K
#         if p_value < best_pval_for_k:
#             best_pval_for_k = p_value
#             best_state_for_k = state
        
#         # Update overall best
#         if p_value < best_overall_pval:
#             best_overall_pval = p_value
#             best_k_states = K_STATES
#             best_state_idx = state
    
#     # Store the best feature for this K
#     best_dwell_features_per_k[K_STATES] = {
#         'features': subject_features[:, best_state_for_k],  # Only the best state
#         'subjects': processed_subjects,
#         'best_state': best_state_for_k,
#         'best_pval': best_pval_for_k
#     }
    
#     print(f"  Best for K={K_STATES}: State {best_state_for_k}, p-value = {best_pval_for_k:.6f}")

# print(f"\nOverall best dwell-time feature: K={best_k_states}, State={best_state_idx}, p={best_overall_pval:.6f}")

# # Add only the best dwell-time feature for each K to the dataframe
# for K_STATES in K_STATES_LIST:
#     features_data = best_dwell_features_per_k[K_STATES]
#     best_features = features_data['features']
#     processed_subjects = features_data['subjects']
#     best_state = features_data['best_state']
#     best_pval = features_data['best_pval']
    
#     # Initialize column for this K value (only one column per K)
#     col_name = f'fMRI_dwell_K{K_STATES}_best'
#     final_df[col_name] = np.nan
    
#     # Add metadata columns for this K
#     final_df[f'fMRI_K{K_STATES}_best_state'] = best_state
#     final_df[f'fMRI_K{K_STATES}_best_pval'] = best_pval
    
#     # Fill in the dwell-time values for subjects that have fMRI data
#     for i, subject_id in enumerate(processed_subjects):
#         # Convert subject_id to participant_id format
#         participant_id = f'sub-{subject_id}'
        
#         if participant_id in participant_to_idx:
#             df_idx = participant_to_idx[participant_id]
#             final_df.loc[final_df.index[df_idx], col_name] = best_features[i]

# # Add group mapping
# group_mapping = {'N': 0, 'A+P-': 1, 'A+P+': 2}
# final_df['Group'] = final_df['Risk_Group'].map(group_mapping)

# # Add overall best feature indicator columns
# final_df['overall_best_K'] = best_k_states
# final_df['overall_best_state'] = best_state_idx
# final_df['overall_best_pval'] = best_overall_pval

# # Save to CSV
# output_filename = f"fMRI_best_dwell_features_{classes[0]}_vs_{classes[1].replace('+', 'plus').replace('-', 'minus')}.csv"
# final_df.to_csv(output_filename, index=False)

# print(f"\nDataframe saved to: {output_filename}")
# print(f"Shape: {final_df.shape}")

# # Show the new columns
# dwell_cols = [col for col in final_df.columns if 'fMRI_dwell' in col]
# metadata_cols = [col for col in final_df.columns if 'fMRI_K' in col and ('best_state' in col or 'best_pval' in col)]

# print(f"\nDwell-time feature columns (best per K):")
# for col in dwell_cols:
#     print(f"  {col}")

# print(f"\nMetadata columns:")
# for col in metadata_cols:
#     print(f"  {col}")

# print(f"\nSample of data:")
# sample_cols = ['participant_id', 'Risk_Group', 'Group'] + dwell_cols[:3] + metadata_cols[:3]
# print(final_df[sample_cols].head())

In [ ]:
# from sklearn.cluster import KMeans
# import pandas as pd
# import numpy as np
# from scipy.stats import ttest_ind, f_oneway, mannwhitneyu, kruskal, shapiro, levene

# # Load your data (assuming you have already loaded it)
# from utils import load_multimodal_data

# # Parameters
# WIN_STEP = 5
# TR = 0.8
# STEP_SEC = WIN_STEP * TR
# K_STATES_LIST = list(range(5, 11))
# SEED = 100

# # Load data for specific classes (example: N vs A+P-)
# classes = ("all")
# multimodal_data = load_multimodal_data(classes=classes, verbose=True)
# fmri_data = multimodal_data['fmri']

# def compute_subject_dwell_times_fmri(X_data, y_data, sub_ids_data, target_subjects, kmeans_model, k_states):
#     subject_features, subject_labels, processed_subjects = [], [], []
    
#     # Get unique subjects only
#     unique_subjects = np.unique(target_subjects)
    
#     for subject in unique_subjects:
#         subject_mask = sub_ids_data == subject
#         if not np.any(subject_mask): 
#             continue
        
#         # Get all time points for this subject
#         subject_fmri = X_data[subject_mask]
#         subject_label = y_data[subject_mask][0]  # All samples for same subject have same label
        
#         # Predict cluster for each time point
#         subject_preds = kmeans_model.predict(subject_fmri)
        
#         # Calculate dwell times (time spent in each state)
#         subject_dwell_times = np.bincount(subject_preds, minlength=k_states) * STEP_SEC
        
#         subject_features.append(subject_dwell_times)
#         subject_labels.append(subject_label)
#         processed_subjects.append(subject)
    
#     return np.array(subject_features), np.array(subject_labels), processed_subjects

# def choose_statistical_test(groups):
#     if len(groups) < 2 or any(g.size == 0 for g in groups):
#         return 'None', 1.0, 0.0
    
#     # Check normality and equal variances
#     try:
#         normality_ok = all(shapiro(g)[1] > 0.05 for g in groups if len(g) >= 3)
#         equal_var_ok = levene(*groups)[1] > 0.05
        
#         if normality_ok and equal_var_ok:
#             # Use parametric tests
#             if len(groups) == 2:
#                 stat, p_val = ttest_ind(*groups)
#                 return 't-test', p_val, stat
#             else:
#                 stat, p_val = f_oneway(*groups)
#                 return 'ANOVA', p_val, stat
#         else:
#             # Use non-parametric tests
#             if len(groups) == 2:
#                 stat, p_val = mannwhitneyu(*groups)
#                 return 'Mann-Whitney', p_val, stat
#             else:
#                 stat, p_val = kruskal(*groups)
#                 return 'Kruskal-Wallis', p_val, stat
#     except:
#         return 'None', 1.0, 0.0

# # Load participant data and create risk groups
# import os
# EEG_data_files = "/data/s.dharia-ra/PEARL/final_multi_modal/data/EEG_data/EEG_data_plv"
# EEG_subjects = [f.split('_')[0] for f in os.listdir(EEG_data_files) if f.endswith('.npz')]
# EEG_subjects = set(EEG_subjects)

# fMRI_data_files = "/data/s.dharia-ra/PEARL/final_multi_modal/data/fMRI_data"
# fMRI_subjects = [f.split('_')[0] for f in os.listdir(fMRI_data_files) if f.endswith('.npz')]
# fMRI_subjects = set(fMRI_subjects)

# common_subjects = EEG_subjects.intersection(fMRI_subjects)
# print(f"Number of subjects with both EEG and fMRI data: {len(common_subjects)}")

# df = pd.read_csv("./participants.tsv", sep="\t")

# # Create risk group classification
# conditions = [
#     (~df['APOE_haplotype'].str.contains('e4', case=False)) & (df['PICALM_rs3851179'] != 'G/G'),
#     (df['APOE_haplotype'].str.contains('e4', case=False)) & (df['PICALM_rs3851179'] != 'G/G'),
#     (df['APOE_haplotype'].str.contains('e4', case=False)) & (df['PICALM_rs3851179'] == 'G/G')
# ]

# choices = ['N', 'A+P-', 'A+P+']

# df['Risk_Group'] = np.select(conditions, choices, default='Excluded')

# # Remove APOE e2-only carriers from N group
# df.loc[(df['APOE_haplotype'].str.contains('e2', case=False)) & 
#        (~df['APOE_haplotype'].str.contains('e4', case=False)) & 
#        (df['Risk_Group'] == 'N'), 'Risk_Group'] = 'Excluded'

# # Filter to only include the specified classes and subjects with both EEG and fMRI data
# if classes == ("all"):
#     study_df = df[df['Risk_Group'].isin(['N', 'A+P-', 'A+P+'])].copy()
# else:
#     study_df = df[df['Risk_Group'].isin(list(classes))].copy()

# study_df['Has_EEG_fMRI'] = study_df['participant_id'].apply(lambda x: x in common_subjects)
# final_df = study_df[study_df['Has_EEG_fMRI']].copy()

# print(f"Final dataframe shape: {final_df.shape}")
# print(f"Classes distribution: {final_df['Risk_Group'].value_counts()}")

# # Create a mapping from participant_id to dataframe index for easy lookup
# participant_to_idx = {pid: idx for idx, pid in enumerate(final_df['participant_id'])}

# # === MODIFIED: Storage for ONLY BEST cluster windows per K ===
# best_cluster_windows_by_k = {}     # Storage for best cluster per K
# all_best_cluster_windows = []      # Final list with all best clusters

# # Find best dwell-time feature with lowest p-value and save ONLY best cluster windows
# print("Searching for BEST dwell-time features and saving BEST cluster windows...")
# best_overall_pval = 1.0
# best_k_states = None
# best_state_idx = None

# # Dictionary to store the best (most significant) dwell-time feature for each K
# best_dwell_features_per_k = {}

# for K_STATES in K_STATES_LIST:
#     print(f"Testing K_STATES = {K_STATES}")
    
#     # Fit KMeans
#     kmeans = KMeans(n_clusters=K_STATES, random_state=SEED, n_init="auto", max_iter=300)
#     kmeans.fit(fmri_data['X'])
    
#     # Get cluster predictions for all windows
#     all_cluster_predictions = kmeans.predict(fmri_data['X'])
    
#     # Get unique subjects
#     unique_subjects = np.unique(fmri_data['sub_ids'])
    
#     # Calculate dwell-time features for unique subjects only
#     subject_features, subject_labels, processed_subjects = compute_subject_dwell_times_fmri(
#         X_data=fmri_data['X'],
#         y_data=fmri_data['y'],
#         sub_ids_data=fmri_data['sub_ids'],
#         target_subjects=unique_subjects,
#         kmeans_model=kmeans,
#         k_states=K_STATES
#     )
    
#     print(f"Subject features shape: {subject_features.shape}, Subject labels shape: {subject_labels.shape}")
    
#     # Find the BEST state (lowest p-value) for this K
#     best_pval_for_k = 1.0
#     best_state_for_k = None
    
#     for state in range(K_STATES):
#         groups = [subject_features[subject_labels == label, state] for label in np.unique(subject_labels)]
#         test_name, p_value, stat = choose_statistical_test(groups)
        
#         print(f"  State {state}: {test_name}, p-value = {p_value:.6f}")
        
#         # Update best for this K
#         if p_value < best_pval_for_k:
#             best_pval_for_k = p_value
#             best_state_for_k = state
        
#         # Update overall best
#         if p_value < best_overall_pval:
#             best_overall_pval = p_value
#             best_k_states = K_STATES
#             best_state_idx = state
    
#     # === SAVE ONLY THE BEST CLUSTER WINDOWS FOR THIS K ===
#     if best_state_for_k is not None:
#         # Get windows belonging to the BEST cluster only
#         best_cluster_mask = all_cluster_predictions == best_state_for_k
#         best_cluster_windows = fmri_data['X'][best_cluster_mask]
        
#         # Store best cluster info for this K
#         best_cluster_windows_by_k[K_STATES] = {
#             'windows': best_cluster_windows,
#             'n_windows': len(best_cluster_windows),
#             'best_state': best_state_for_k,
#             'best_pval': best_pval_for_k,
#             'window_indices': np.where(best_cluster_mask)[0]
#         }
        
#         # Add to the comprehensive list
#         all_best_cluster_windows.extend(best_cluster_windows)
        
#         print(f"  *** BEST CLUSTER for K={K_STATES}: State {best_state_for_k}, p={best_pval_for_k:.6f}, {len(best_cluster_windows)} windows")
    
#     # Store the best feature for this K (for backward compatibility with dataframe)
#     best_dwell_features_per_k[K_STATES] = {
#         'features': subject_features[:, best_state_for_k],  # Only the best state
#         'subjects': processed_subjects,
#         'best_state': best_state_for_k,
#         'best_pval': best_pval_for_k
#     }
    
#     print(f"  Best for K={K_STATES}: State {best_state_for_k}, p-value = {best_pval_for_k:.6f}")

# print(f"\nOverall best dwell-time feature: K={best_k_states}, State={best_state_idx}, p={best_overall_pval:.6f}")

# # === CREATE LIST OF BEST CLUSTER WINDOWS ONLY ===
# print("\n" + "="*80)
# print("CREATING LIST OF BEST CLUSTER WINDOWS ONLY")
# print("="*80)

# print(f"Total number of BEST cluster windows across all K values: {len(all_best_cluster_windows)}")

# # Summary of best clusters per K
# print(f"\n" + "="*60)
# print("SUMMARY OF BEST CLUSTERS PER K")
# print("="*60)

# for k_val in K_STATES_LIST:
#     if k_val in best_cluster_windows_by_k:
#         cluster_info = best_cluster_windows_by_k[k_val]
#         best_state = cluster_info['best_state']
#         best_pval = cluster_info['best_pval']
#         n_windows = cluster_info['n_windows']
#         print(f"K={k_val}: Best State {best_state}, p={best_pval:.6f}, {n_windows} windows")
#     else:
#         print(f"K={k_val}: No valid clusters found")

# # === SAVE BEST CLUSTER WINDOWS ===
# print(f"\n" + "="*60)
# print("SAVING BEST CLUSTER WINDOWS")
# print("="*60)

# # Save individual best cluster windows for each K
# for k_val, cluster_info in best_cluster_windows_by_k.items():
#     filename = f"best_cluster_K{k_val}_state{cluster_info['best_state']}_p{cluster_info['best_pval']:.6f}.npz"
#     np.savez_compressed(filename, 
#                        windows=cluster_info['windows'],
#                        p_value=cluster_info['best_pval'],
#                        K=k_val,
#                        state=cluster_info['best_state'],
#                        n_windows=cluster_info['n_windows'])
#     print(f"Saved: {filename} ({cluster_info['n_windows']} windows)")

# # Save comprehensive collection of all best cluster windows
# np.savez_compressed('all_best_cluster_windows.npz',
#                    all_best_windows=np.array(all_best_cluster_windows))

# print(f"\nSaved comprehensive collection: all_best_cluster_windows.npz")

# # === FINAL OUTPUT: The list you requested ===
# print(f"\n" + "="*80)
# print("FINAL OUTPUT: LIST OF BEST CLUSTER WINDOWS")
# print("="*80)

# # Main list containing ONLY the best cluster windows
# print(f"📋 Main list: 'all_best_cluster_windows' contains {len(all_best_cluster_windows)} windows")
# print(f"   Type: List of numpy arrays")
# print(f"   Shape of each window: {all_best_cluster_windows[0].shape if all_best_cluster_windows else 'N/A'}")

# # Dictionary for easy access by K value
# print(f"\n📋 Dictionary: 'best_cluster_windows_by_k' organized by K values:")
# for k_val, cluster_info in best_cluster_windows_by_k.items():
#     print(f"   K={k_val}: Best cluster with {cluster_info['n_windows']} windows (p={cluster_info['best_pval']:.6f})")

# # Add the dataframe creation (original functionality)
# print(f"\n" + "="*60)
# print("CREATING ORIGINAL DATAFRAME WITH BEST FEATURES")
# print("="*60)

# # Add only the best dwell-time feature for each K to the dataframe
# for K_STATES in K_STATES_LIST:
#     features_data = best_dwell_features_per_k[K_STATES]
#     best_features = features_data['features']
#     processed_subjects = features_data['subjects']
#     best_state = features_data['best_state']
#     best_pval = features_data['best_pval']
    
#     # Initialize column for this K value (only one column per K)
#     col_name = f'fMRI_dwell_K{K_STATES}_best'
#     final_df[col_name] = np.nan
    
#     # Add metadata columns for this K
#     final_df[f'fMRI_K{K_STATES}_best_state'] = best_state
#     final_df[f'fMRI_K{K_STATES}_best_pval'] = best_pval
    
#     # Fill in the dwell-time values for subjects that have fMRI data
#     for i, subject_id in enumerate(processed_subjects):
#         # Convert subject_id to participant_id format
#         participant_id = f'sub-{subject_id}'
        
#         if participant_id in participant_to_idx:
#             df_idx = participant_to_idx[participant_id]
#             final_df.loc[final_df.index[df_idx], col_name] = best_features[i]

# # Add group mapping
# group_mapping = {'N': 0, 'A+P-': 1, 'A+P+': 2}
# final_df['Group'] = final_df['Risk_Group'].map(group_mapping)

# # Add overall best feature indicator columns
# final_df['overall_best_K'] = best_k_states
# final_df['overall_best_state'] = best_state_idx
# final_df['overall_best_pval'] = best_overall_pval

# # Save to CSV
# output_filename = "fMRI_best_dwell_features_all_groups.csv"
# final_df.to_csv(output_filename, index=False)

# print(f"\nDataframe saved to: {output_filename}")
# print(f"Shape: {final_df.shape}")

# direction aware

In [ ]:
from sklearn.cluster import KMeans
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind, f_oneway, mannwhitneyu, kruskal, shapiro, levene
from utils import load_multimodal_data
import os

# Parameters
WIN_STEP = 5
TR = 0.8
STEP_SEC = WIN_STEP * TR
K_STATES_LIST = list(range(4, 10))
SEED = 100
P_VALUE_THRESHOLD = 0.05  # Significance threshold

# Load data
classes = ("all")
multimodal_data = load_multimodal_data(classes=classes, verbose=True)
fmri_data = multimodal_data['fmri']

def compute_subject_dwell_times_fmri(X_data, y_data, sub_ids_data, target_subjects, kmeans_model, k_states):
    subject_features, subject_labels, processed_subjects = [], [], []
    unique_subjects = np.unique(target_subjects)
    
    for subject in unique_subjects:
        subject_mask = sub_ids_data == subject
        if not np.any(subject_mask): 
            continue
        
        subject_fmri = X_data[subject_mask]
        subject_label = y_data[subject_mask][0]
        subject_preds = kmeans_model.predict(subject_fmri)
        subject_dwell_times = np.bincount(subject_preds, minlength=k_states) * STEP_SEC
        
        subject_features.append(subject_dwell_times)
        subject_labels.append(subject_label)
        processed_subjects.append(subject)
    
    return np.array(subject_features), np.array(subject_labels), processed_subjects

def choose_statistical_test(groups):
    if len(groups) < 2 or any(g.size == 0 for g in groups):
        return 'None', 1.0, 0.0
    
    try:
        normality_ok = all(shapiro(g)[1] > 0.05 for g in groups if len(g) >= 3)
        equal_var_ok = levene(*groups)[1] > 0.05
        
        if normality_ok and equal_var_ok:
            if len(groups) == 2:
                stat, p_val = ttest_ind(*groups)
                return 't-test', p_val, stat
            else:
                stat, p_val = f_oneway(*groups)
                return 'ANOVA', p_val, stat
        else:
            if len(groups) == 2:
                stat, p_val = mannwhitneyu(*groups)
                return 'Mann-Whitney', p_val, stat
            else:
                stat, p_val = kruskal(*groups)
                return 'Kruskal-Wallis', p_val, stat
    except:
        return 'None', 1.0, 0.0

def determine_dominant_group(subject_features, subject_labels, state):
    """Determine which risk group has highest mean dwell time for this state"""
    unique_labels = np.unique(subject_labels)
    label_to_group = {0: 'N', 1: 'AposP minus', 2: 'AposPpos'}
    
    group_means = {}
    for label in unique_labels:
        group_dwell_times = subject_features[subject_labels == label, state]
        group_means[label] = np.mean(group_dwell_times)
    
    # Find label with highest mean
    dominant_label = max(group_means, key=group_means.get)
    dominant_group = label_to_group[dominant_label]
    
    # Create summary string
    means_str = "_".join([f"{label_to_group[lbl]}{group_means[lbl]:.1f}" 
                          for lbl in sorted(unique_labels)])
    
    return dominant_group, group_means, means_str

# Load participant data
EEG_data_files = "/data/s.dharia-ra/PEARL/final_multi_modal/data/EEG_data/EEG_data_plv"
EEG_subjects = set([f.split('_')[0] for f in os.listdir(EEG_data_files) if f.endswith('.npz')])

fMRI_data_files = "/data/s.dharia-ra/PEARL/final_multi_modal/data/fMRI_data"
fMRI_subjects = set([f.split('_')[0] for f in os.listdir(fMRI_data_files) if f.endswith('.npz')])

common_subjects = EEG_subjects.intersection(fMRI_subjects)
print(f"Number of subjects with both EEG and fMRI data: {len(common_subjects)}")

df = pd.read_csv("./participants.tsv", sep="\t")

# Create risk groups
conditions = [
    (~df['APOE_haplotype'].str.contains('e4', case=False)) & (df['PICALM_rs3851179'] != 'G/G'),
    (df['APOE_haplotype'].str.contains('e4', case=False)) & (df['PICALM_rs3851179'] != 'G/G'),
    (df['APOE_haplotype'].str.contains('e4', case=False)) & (df['PICALM_rs3851179'] == 'G/G')
]
choices = ['N', 'A+P-', 'A+P+']
df['Risk_Group'] = np.select(conditions, choices, default='Excluded')

df.loc[(df['APOE_haplotype'].str.contains('e2', case=False)) & 
       (~df['APOE_haplotype'].str.contains('e4', case=False)) & 
       (df['Risk_Group'] == 'N'), 'Risk_Group'] = 'Excluded'

study_df = df[df['Risk_Group'].isin(['N', 'A+P-', 'A+P+'])].copy()
study_df['Has_EEG_fMRI'] = study_df['participant_id'].apply(lambda x: x in common_subjects)
final_df = study_df[study_df['Has_EEG_fMRI']].copy()

print(f"Final dataframe shape: {final_df.shape}")
print(f"Classes distribution: {final_df['Risk_Group'].value_counts()}")

participant_to_idx = {pid: idx for idx, pid in enumerate(final_df['participant_id'])}

# === FIND ALL SIGNIFICANT CLUSTERS ===
print("\n" + "="*80)
print("FINDING ALL SIGNIFICANT CLUSTERS ACROSS K VALUES")
print("="*80)

significant_clusters = []  # Store all significant clusters
all_dwell_features = {}    # Store all features for dataframe

for K_STATES in K_STATES_LIST:
    print(f"\nTesting K_STATES = {K_STATES}")
    print("="*60)
    
    # Fit KMeans
    kmeans = KMeans(n_clusters=K_STATES, random_state=SEED, n_init="auto", max_iter=300)
    kmeans.fit(fmri_data['X'])
    
    # Get cluster predictions
    all_cluster_predictions = kmeans.predict(fmri_data['X'])
    unique_subjects = np.unique(fmri_data['sub_ids'])
    
    # Calculate dwell-time features
    subject_features, subject_labels, processed_subjects = compute_subject_dwell_times_fmri(
        X_data=fmri_data['X'],
        y_data=fmri_data['y'],
        sub_ids_data=fmri_data['sub_ids'],
        target_subjects=unique_subjects,
        kmeans_model=kmeans,
        k_states=K_STATES
    )
    
    # Test each state
    for state in range(K_STATES):
        groups = [subject_features[subject_labels == label, state] 
                 for label in np.unique(subject_labels)]
        test_name, p_value, stat = choose_statistical_test(groups)
        
        # Check if significant
        is_significant = p_value < P_VALUE_THRESHOLD
        significance_marker = "***" if is_significant else ""
        
        print(f"  State {state}: {test_name}, p={p_value:.6f} {significance_marker}")
        
        # If significant, save this cluster
        if is_significant:
            # Determine which group has highest dwell time
            dominant_group, group_means, means_str = determine_dominant_group(
                subject_features, subject_labels, state
            )
            
            # Get windows for this cluster
            cluster_mask = all_cluster_predictions == state
            cluster_windows = fmri_data['X'][cluster_mask]
            
            # Store cluster info
            cluster_info = {
                'K': K_STATES,
                'state': state,
                'p_value': p_value,
                'test': test_name,
                'dominant_group': dominant_group,
                'group_means': group_means,
                'windows': cluster_windows,
                'n_windows': len(cluster_windows),
                'subject_features': subject_features[:, state],
                'processed_subjects': processed_subjects
            }
            
            significant_clusters.append(cluster_info)
            
            print(f"    ✓ SIGNIFICANT! Dominant group: {dominant_group}")
            print(f"      Group means: {group_means}")
            print(f"      Windows: {len(cluster_windows)}")

print(f"\n" + "="*80)
print(f"FOUND {len(significant_clusters)} SIGNIFICANT CLUSTERS")
print("="*80)

# === SAVE ALL SIGNIFICANT CLUSTERS ===
print("\n" + "="*80)
print("SAVING SIGNIFICANT CLUSTERS")
print("="*80)

output_dir = "fMRI_significant_clusters"
os.makedirs(output_dir, exist_ok=True)

for idx, cluster in enumerate(significant_clusters, 1):
    # Create descriptive filename
    filename = (f"cluster_K{cluster['K']}_state{cluster['state']}_"
               f"{cluster['dominant_group']}_dominant_"
               f"p{cluster['p_value']:.6f}.npz")
    
    filepath = os.path.join(output_dir, filename)
    
    # Save cluster data
    np.savez_compressed(
        filepath,
        windows=cluster['windows'],
        K=cluster['K'],
        state=cluster['state'],
        p_value=cluster['p_value'],
        test=cluster['test'],
        dominant_group=cluster['dominant_group'],
        group_means=cluster['group_means'],
        n_windows=cluster['n_windows']
    )
    
    print(f"{idx:2d}. {filename}")
    print(f"    Windows: {cluster['n_windows']}, p={cluster['p_value']:.6f}")

# === CREATE COMPREHENSIVE DATAFRAME ===
print("\n" + "="*80)
print("CREATING DATAFRAME WITH ALL SIGNIFICANT FEATURES")
print("="*80)

for idx, cluster in enumerate(significant_clusters):
    K = cluster['K']
    state = cluster['state']
    dominant_group = cluster['dominant_group']
    p_val = cluster['p_value']
    
    # Create column name
    col_name = f'fMRI_K{K}_state{state}_{dominant_group}'
    final_df[col_name] = np.nan
    
    # Add metadata
    final_df[f'{col_name}_pval'] = p_val
    final_df[f'{col_name}_test'] = cluster['test']
    
    # Fill in values
    for i, subject_id in enumerate(cluster['processed_subjects']):
        participant_id = f'sub-{subject_id}'
        if participant_id in participant_to_idx:
            df_idx = participant_to_idx[participant_id]
            final_df.loc[final_df.index[df_idx], col_name] = cluster['subject_features'][i]

# Add group mapping
group_mapping = {'N': 0, 'A+P-': 1, 'A+P+': 2}
final_df['Group'] = final_df['Risk_Group'].map(group_mapping)

# Save dataframe
output_csv = "fMRI_all_significant_features.csv"
final_df.to_csv(output_csv, index=False)

print(f"\n✓ Saved dataframe: {output_csv}")
print(f"  Shape: {final_df.shape}")
print(f"  Significant feature columns: {len(significant_clusters)}")

# === SUMMARY REPORT ===
print("\n" + "="*80)
print("SUMMARY REPORT")
print("="*80)

print(f"\nTotal significant clusters found: {len(significant_clusters)}")

# Group by dominant group
from collections import Counter
dominant_counts = Counter([c['dominant_group'] for c in significant_clusters])

print(f"\nClusters by dominant group:")
for group, count in dominant_counts.items():
    print(f"  {group}: {count} clusters")

print(f"\nClusters by K value:")
k_counts = Counter([c['K'] for c in significant_clusters])
for k, count in sorted(k_counts.items()):
    print(f"  K={k}: {count} significant clusters")

print(f"\nP-value range:")
p_values = [c['p_value'] for c in significant_clusters]
print(f"  Min: {min(p_values):.6f}")
print(f"  Max: {max(p_values):.6f}")
print(f"  Mean: {np.mean(p_values):.6f}")

print("\n" + "="*80)
print("✅ COMPLETE!")
print("="*80)

# window analysis

In [ ]:
import numpy as np
import os 

path = "fMRI_analysis_windows_new/N_group_higher"

# find npz files in the path
npz_files = [f for f in os.listdir(path) if f.endswith('.npz')]

mean_windows = []
for file in npz_files:
    data = np.load(os.path.join(path, file))
    windows = data['windows']
    mean_window = np.mean(windows, axis=0)
    mean_windows.append(mean_window)
mean_windows = np.array(mean_windows)
print(f"Mean windows shape: {mean_windows.shape}")

# all_mean 

all_mean = np.mean(mean_windows, axis=0)
print(f"All mean shape: {all_mean.shape}")

# get the top 5 indices with highest values
top_5_indices = np.argsort(all_mean)[-5:][::-1]
print(f"Top 5 indices: {top_5_indices}")



In [ ]:
import nibabel as nib
from nilearn import datasets
import templateflow.api as tf

atlas_paths = tf.get(
    template='MNI152NLin2009cAsym', atlas='Schaefer2018',
    desc='100Parcels7Networks', extension='.nii.gz'
)

atlas_path = atlas_paths[0] if isinstance(atlas_paths, list) else atlas_paths
atlas_img = nib.load(atlas_path)
atlas_data = atlas_img.get_fdata()

# atlas names
labels = datasets.fetch_atlas_schaefer_2018(n_rois=100, yeo_networks=7)
atlas_labels = labels['labels']
print(f"Atlas labels: {atlas_labels}")

In [ ]:
import numpy as np
import os 
from nilearn import datasets

# Your existing code
path = "fMRI_significant_clusters/risk_group_higher"


# path = "fMRI_significant_clusters/N_group_higher"
npz_files = [f for f in os.listdir(path) if f.endswith('.npz')]

mean_windows = []
for file in npz_files:
    data = np.load(os.path.join(path, file))
    windows = data['windows']
    mean_window = np.median(windows, axis=0)
    mean_windows.append(mean_window)
mean_windows = np.array(mean_windows)

all_mean = np.mean(mean_windows, axis=0)

print(f"Total features in flattened array: {len(all_mean)}")
print(f"Expected for 100x100 upper triangular (no diagonal): {100*99//2}")
print(f"Expected for 100x100 lower triangular (no diagonal): {100*99//2}")

# Get top 10 indices
top_10_indices = np.argsort(all_mean)[-10:][::-1]
print(f"\nTop 10 indices: {top_10_indices}")

# Load atlas labels
labels = datasets.fetch_atlas_schaefer_2018(n_rois=100, yeo_networks=7)
atlas_labels = [label.decode('utf-8') for label in labels['labels']]

# CORRECTED FUNCTION with validation
def flat_to_matrix_coords(flat_idx, matrix_size=100):
    """
    Convert flattened index back to matrix coordinates
    Assumes upper triangular matrix (excluding diagonal) was flattened
    """
    idx = 0
    for i in range(matrix_size):
        for j in range(i + 1, matrix_size):  # Upper triangular, excluding diagonal
            if idx == flat_idx:
                return i, j
            idx += 1
    
    return None, None

# Alternative mathematical formula (faster and more reliable)
def flat_to_matrix_coords_formula(k, n=100):
    """
    Mathematical formula to convert flattened upper triangular index to (i,j)
    For upper triangular matrix WITHOUT diagonal
    """
    i = n - 2 - int(np.floor(np.sqrt(-8*k + 4*n*(n-1) - 7)/2.0 - 0.5))
    j = k + i + 1 - n*(n-1)//2 + (n-i)*((n-i)-1)//2
    return i, j

# VERIFICATION: Test both methods match
print("\n" + "="*80)
print("VERIFICATION: Comparing both methods for consistency")
print("="*80)

for test_idx in [0, 100, 1000, 2000, 4949]:  # Test various indices
    i1, j1 = flat_to_matrix_coords(test_idx)
    i2, j2 = flat_to_matrix_coords_formula(test_idx)
    match = "✓" if (i1, j1) == (i2, j2) else "✗"
    print(f"Index {test_idx:4d}: Loop method ({i1:2d},{j1:2d}) | Formula method ({i2:2d},{j2:2d}) {match}")

# VALIDATION: Reconstruct and verify
print("\n" + "="*80)
print("VALIDATION: Verify index reconstruction")
print("="*80)

def matrix_coords_to_flat(i, j, n=100):
    """Convert matrix coordinates back to flat index"""
    if i >= j:
        i, j = j, i  # Ensure i < j for upper triangular
    flat_idx = i * n - (i * (i + 1)) // 2 + j - i - 1
    return flat_idx

# Test round-trip conversion
for test_idx in top_10_indices[:3]:
    i, j = flat_to_matrix_coords_formula(test_idx)
    reconstructed_idx = matrix_coords_to_flat(i, j)
    match = "✓" if test_idx == reconstructed_idx else "✗"
    print(f"Original idx: {test_idx} → ({i},{j}) → Reconstructed idx: {reconstructed_idx} {match}")

# NOW EXTRACT THE TOP 10 PAIRS WITH FULL VALIDATION
print("\n" + "="*80)
print("TOP 10 FUNCTIONAL CONNECTIVITY PAIRS (VERIFIED)")
print("="*80)

for rank, flat_idx in enumerate(top_10_indices, 1):
    # Use the formula method (more reliable)
    i, j = flat_to_matrix_coords_formula(flat_idx)
    
    if i is not None and j is not None and 0 <= i < 100 and 0 <= j < 100:
        # Get region names
        region_1 = atlas_labels[i]
        region_2 = atlas_labels[j]
        
        # Get the actual correlation value
        correlation_value = all_mean[flat_idx]
        
        print(f"\nRank {rank}: Flat Index {flat_idx}")
        print(f"  Matrix position: ({i}, {j})")
        print(f"  Region 1 (ROI {i}): {region_1}")
        print(f"  Region 2 (ROI {j}): {region_2}")
        print(f"  Mean correlation: {correlation_value:.6f}")
        
        # Verify this is upper triangular
        if i >= j:
            print(f"  ⚠️ WARNING: Not upper triangular! i={i} should be < j={j}")
    else:
        print(f"Rank {rank}: Invalid coordinates for index {flat_idx}")

# COMPACT SUMMARY
print("\n" + "="*80)
print("COMPACT SUMMARY - Top 10 FC pairs")
print("="*80)

for rank, flat_idx in enumerate(top_10_indices, 1):
    i, j = flat_to_matrix_coords_formula(flat_idx)
    
    if i is not None and j is not None and 0 <= i < 100 and 0 <= j < 100:
        region_1_short = atlas_labels[i].replace('7Networks_', '').replace('_', ' ')
        region_2_short = atlas_labels[j].replace('7Networks_', '').replace('_', ' ')
        correlation_value = all_mean[flat_idx]
        
        print(f"{rank:2d}. ROI({i:2d},{j:2d}): {region_1_short} <-> {region_2_short} (r={correlation_value:.4f})")

# FINAL CHECK: Print some statistics
print("\n" + "="*80)
print("FINAL STATISTICS")
print("="*80)
print(f"Mean correlation across all pairs: {np.mean(all_mean):.4f}")
print(f"Std correlation across all pairs: {np.std(all_mean):.4f}")
print(f"Min correlation: {np.min(all_mean):.4f}")
print(f"Max correlation: {np.max(all_mean):.4f}")
print(f"Top 10 correlation range: {all_mean[top_10_indices[-1]]:.4f} to {all_mean[top_10_indices[0]]:.4f}")

In [ ]:
import numpy as np
import templateflow.api as tf
import nibabel as nib
from nilearn import image
from scipy import ndimage
from nilearn.plotting import view_connectome
from pathlib import Path

# 1. LOAD SCHAEFER ATLAS AND GET COORDINATES
print("Loading Schaefer atlas coordinates...")

atlas_paths = tf.get(
    template='MNI152NLin2009cAsym', atlas='Schaefer2018',
    desc='100Parcels7Networks', extension='.nii.gz'
)

atlas_path = atlas_paths[0] if isinstance(atlas_paths, list) else atlas_paths
atlas_img = nib.load(atlas_path)
atlas_data = atlas_img.get_fdata()

def get_region_coordinates(atlas_data, atlas_img):
    """Extract center of mass coordinates for each region"""
    coordinates = []
    unique_labels = np.unique(atlas_data)[1:]  # Skip 0 (background)
    
    for label in unique_labels:
        region_mask = (atlas_data == label)
        com_voxel = ndimage.center_of_mass(region_mask)
        com_mni = image.coord_transform(
            com_voxel[0], com_voxel[1], com_voxel[2], 
            atlas_img.affine
        )
        coordinates.append([com_mni[0], com_mni[1], com_mni[2]])
    
    return np.array(coordinates)

coordinates = get_region_coordinates(atlas_data, atlas_img)
print(f"✓ Loaded {len(coordinates)} region coordinates")

# 2. CREATE ADJACENCY MATRIX FROM YOUR TOP 10 CONNECTIONS
print("\nCreating adjacency matrix from top 10 connections...")

n_rois = 100
adj_matrix = np.zeros((n_rois, n_rois))

# Use your already computed top_10_indices and all_mean
for flat_idx in top_10_indices:
    i, j = flat_to_matrix_coords_formula(flat_idx)
    correlation_value = all_mean[flat_idx]
    
    # Set symmetric values in adjacency matrix
    adj_matrix[i, j] = correlation_value
    adj_matrix[j, i] = correlation_value

print(f"✓ Created adjacency matrix with {np.count_nonzero(adj_matrix)} edges (top 10 pairs)")

# 3. CREATE INTERACTIVE VISUALIZATION
print("\n🌐 Creating interactive brain connectome...")

interactive_view = view_connectome(
    adj_matrix, 
    coordinates, 
    node_size=5,
    node_color="gray",
    edge_cmap='Blues',
    colorbar=False,
)

# Save interactive view
output_dir = Path("fMRI_analysis_windows_new")
output_interactive = output_dir / "top10_connections_interactive.html"
interactive_view.save_as_html(str(output_interactive))

print(f"✓ Saved interactive view to: {output_interactive}")
print(f"\n✅ Done! Open '{output_interactive}' in your browser to view the 3D brain connectome.")

In [ ]:
import numpy as np
import os 
from nilearn import datasets

# Load data
path = "fMRI_analysis_windows_new"
npz_files = [f for f in os.listdir(path) if f.endswith('.npz')]

mean_windows = []
for file in npz_files:
    data = np.load(os.path.join(path, file))
    windows = data['windows']
    mean_window = np.mean(windows, axis=0)
    mean_windows.append(mean_window)
mean_windows = np.array(mean_windows)

all_mean = np.mean(mean_windows, axis=0)
top_10_indices = np.argsort(all_mean)[-10:][::-1]

# Load atlas labels
labels = datasets.fetch_atlas_schaefer_2018(n_rois=100, yeo_networks=7)
atlas_labels = [label.decode('utf-8') for label in labels['labels']]

# CORRECTED: Parse hemisphere info from atlas labels
# Atlas format: '7Networks_LH_Network_Region' or '7Networks_RH_Network_Region'
hemisphere_info = [label.split('_')[1] for label in atlas_labels]  # Changed from [0] to [1]

print("\n" + "="*80)
print("HEMISPHERE DISTRIBUTION IN ATLAS (First 10 ROIs)")
print("="*80)
for i in range(10):
    print(f"ROI {i}: {hemisphere_info[i]} - {atlas_labels[i]}")

print(f"\nLH ROIs: {hemisphere_info.count('LH')}")
print(f"RH ROIs: {hemisphere_info.count('RH')}")

# Conversion function
def flat_to_matrix_coords_formula(k, n=100):
    i = n - 2 - int(np.floor(np.sqrt(-8*k + 4*n*(n-1) - 7)/2.0 - 0.5))
    j = k + i + 1 - n*(n-1)//2 + (n-i)*((n-i)-1)//2
    return i, j

# COMPREHENSIVE ANALYSIS OF TOP 10
print("\n" + "="*80)
print("DETAILED ANALYSIS OF TOP 10 CONNECTIONS")
print("="*80)

connection_types = {
    'LH-LH': 0,
    'RH-RH': 0,
    'LH-RH': 0,
    'RH-LH': 0
}

for rank, flat_idx in enumerate(top_10_indices, 1):
    i, j = flat_to_matrix_coords_formula(flat_idx)
    
    hem_i = hemisphere_info[i]
    hem_j = hemisphere_info[j]
    
    # Determine connection type
    if hem_i == 'LH' and hem_j == 'LH':
        conn_type = 'LH-LH (within left)'
        connection_types['LH-LH'] += 1
    elif hem_i == 'RH' and hem_j == 'RH':
        conn_type = 'RH-RH (within right)'
        connection_types['RH-RH'] += 1
    elif hem_i == 'LH' and hem_j == 'RH':
        conn_type = 'LH-RH (inter-hemispheric)'
        connection_types['LH-RH'] += 1
    else:  # hem_i == 'RH' and hem_j == 'LH'
        conn_type = 'RH-LH (inter-hemispheric)'
        connection_types['RH-LH'] += 1
    
    region_1_short = atlas_labels[i].replace('7Networks_', '').replace('_', ' ')
    region_2_short = atlas_labels[j].replace('7Networks_', '').replace('_', ' ')
    correlation_value = all_mean[flat_idx]
    
    print(f"\n{rank:2d}. [{conn_type}] r={correlation_value:.4f}")
    print(f"    ROI {i:2d} ({hem_i}): {region_1_short}")
    print(f"    ROI {j:2d} ({hem_j}): {region_2_short}")

# SUMMARY OF CONNECTION TYPES
print("\n" + "="*80)
print("CONNECTION TYPE SUMMARY (Top 10)")
print("="*80)
for conn_type, count in connection_types.items():
    print(f"{conn_type}: {count}")

# CHECK IF THIS PATTERN IS CONSISTENT ACROSS MORE CONNECTIONS
print("\n" + "="*80)
print("EXTENDED ANALYSIS: Top 50 Connections")
print("="*80)

top_50_indices = np.argsort(all_mean)[-50:][::-1]

extended_connection_types = {
    'LH-LH': 0,
    'RH-RH': 0,
    'LH-RH': 0,
    'RH-LH': 0
}

for flat_idx in top_50_indices:
    i, j = flat_to_matrix_coords_formula(flat_idx)
    hem_i = hemisphere_info[i]
    hem_j = hemisphere_info[j]
    
    if hem_i == 'LH' and hem_j == 'LH':
        extended_connection_types['LH-LH'] += 1
    elif hem_i == 'RH' and hem_j == 'RH':
        extended_connection_types['RH-RH'] += 1
    elif hem_i == 'LH' and hem_j == 'RH':
        extended_connection_types['LH-RH'] += 1
    else:
        extended_connection_types['RH-LH'] += 1

print(f"Top 50 connection types:")
for conn_type, count in extended_connection_types.items():
    percent = (count / 50) * 100
    print(f"  {conn_type}: {count} ({percent:.1f}%)")

# STATISTICAL SIGNIFICANCE TEST
print("\n" + "="*80)
print("HEMISPHERE BIAS ANALYSIS")
print("="*80)

# Count all inter-hemispheric vs within-hemisphere
inter_hemispheric_top10 = connection_types['LH-RH'] + connection_types['RH-LH']
within_hemispheric_top10 = connection_types['LH-LH'] + connection_types['RH-RH']

inter_hemispheric_top50 = extended_connection_types['LH-RH'] + extended_connection_types['RH-LH']
within_hemispheric_top50 = extended_connection_types['LH-LH'] + extended_connection_types['RH-RH']

print(f"Top 10:")
print(f"  Inter-hemispheric: {inter_hemispheric_top10} ({inter_hemispheric_top10/10*100:.1f}%)")
print(f"  Within-hemispheric: {within_hemispheric_top10} ({within_hemispheric_top10/10*100:.1f}%)")

print(f"\nTop 50:")
print(f"  Inter-hemispheric: {inter_hemispheric_top50} ({inter_hemispheric_top50/50*100:.1f}%)")
print(f"  Within-hemispheric: {within_hemispheric_top50} ({within_hemispheric_top50/50*100:.1f}%)")

# VERIFY INDEXING IS CORRECT
print("\n" + "="*80)
print("INDEXING VERIFICATION")
print("="*80)

# Check if atlas is organized as LH first (0-49), then RH (50-99)
lh_count_first_50 = sum(1 for i in range(50) if hemisphere_info[i] == 'LH')
rh_count_last_50 = sum(1 for i in range(50, 100) if hemisphere_info[i] == 'RH')

print(f"First 50 ROIs (0-49): {lh_count_first_50} LH, {50-lh_count_first_50} RH")
print(f"Last 50 ROIs (50-99): {50-rh_count_last_50} LH, {rh_count_last_50} RH")

if lh_count_first_50 == 50 and rh_count_last_50 == 50:
    print("\n✓ Atlas is organized: ROIs 0-49 are LH, ROIs 50-99 are RH")
    print("✓ Inter-hemispheric connections would have i < 50 and j >= 50")
else:
    print("\n⚠️ Atlas is NOT strictly organized by hemisphere")
    print(f"⚠️ Atlas organization is mixed - LH and RH ROIs are interleaved")

# FINAL DOUBLE-CHECK: Manually verify one connection
print("\n" + "="*80)
print("MANUAL VERIFICATION OF TOP CONNECTION")
print("="*80)

flat_idx = top_10_indices[0]
i, j = flat_to_matrix_coords_formula(flat_idx)

print(f"Flat index: {flat_idx}")
print(f"Matrix coords: ({i}, {j})")
print(f"ROI {i}: {atlas_labels[i]}")
print(f"ROI {j}: {atlas_labels[j]}")
print(f"Correlation: {all_mean[flat_idx]:.4f}")

# Verify by checking what a known LH-LH connection looks like
print(f"\nFor comparison, connection at rank 3:")
test_i, test_j = 17, 18
print(f"ROI {test_i}: {atlas_labels[test_i]} ({hemisphere_info[test_i]})")
print(f"ROI {test_j}: {atlas_labels[test_j]} ({hemisphere_info[test_j]})")

# sMRI 


In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# 1. GET COMMON SUBJECTS (EEG + fMRI)
print("="*80)
print("STEP 1: IDENTIFYING COMMON SUBJECTS")
print("="*80)

# EEG subjects
EEG_data_files = "/data/s.dharia-ra/PEARL/final_multi_modal/data/EEG_data/EEG_data_plv"
EEG_subjects = [f.split('_')[0] for f in os.listdir(EEG_data_files) if f.endswith('.npz')]
EEG_subjects = set(EEG_subjects)

# fMRI subjects
fMRI_data_files = "/data/s.dharia-ra/PEARL/final_multi_modal/data/fMRI_data"
fMRI_subjects = [f.split('_')[0] for f in os.listdir(fMRI_data_files) if f.endswith('.npz')]
fMRI_subjects = set(fMRI_subjects)

# Common subjects
common_subjects = EEG_subjects.intersection(fMRI_subjects)
print(f"✓ EEG subjects: {len(EEG_subjects)}")
print(f"✓ fMRI subjects: {len(fMRI_subjects)}")
print(f"✓ Common subjects (EEG + fMRI): {len(common_subjects)}")

# 2. LOAD PARTICIPANT METADATA AND CREATE RISK GROUPS
print("\n" + "="*80)
print("STEP 2: LOADING PARTICIPANT METADATA & CREATING RISK GROUPS")
print("="*80)

df_participants = pd.read_csv("./participants.tsv", sep="\t")

# Create risk group classification
conditions = [
    (~df_participants['APOE_haplotype'].str.contains('e4', case=False)) & 
    (df_participants['PICALM_rs3851179'] != 'G/G'),
    
    (df_participants['APOE_haplotype'].str.contains('e4', case=False)) & 
    (df_participants['PICALM_rs3851179'] != 'G/G'),
    
    (df_participants['APOE_haplotype'].str.contains('e4', case=False)) & 
    (df_participants['PICALM_rs3851179'] == 'G/G')
]

choices = ['N', 'A+P-', 'A+P+']
df_participants['Risk_Group'] = np.select(conditions, choices, default='Excluded')

# Remove APOE e2-only carriers from N group
df_participants.loc[
    (df_participants['APOE_haplotype'].str.contains('e2', case=False)) & 
    (~df_participants['APOE_haplotype'].str.contains('e4', case=False)) & 
    (df_participants['Risk_Group'] == 'N'), 
    'Risk_Group'
] = 'Excluded'

print(f"✓ Risk group distribution:")
print(df_participants['Risk_Group'].value_counts())

# 3. PROCESS sMRI DATA FILES
print("\n" + "="*80)
print("STEP 3: PROCESSING sMRI DATA FILES")
print("="*80)

sMRI_path = Path("data/MRI_data")
csv_files = list(sMRI_path.glob("*.csv"))

print(f"Found {len(csv_files)} CSV files:")
for f in csv_files:
    print(f"  • {f.name}")

def extract_subject_id(name_string):
    """
    Extract subject ID from name strings like 'sub-01_T1w' or 'sub-01'
    Returns the subject ID in format 'sub-01'
    """
    if pd.isna(name_string):
        return None
    
    # Convert to string and handle different formats
    name_str = str(name_string)
    
    # Try to extract sub-XX pattern
    if 'sub-' in name_str:
        # Split by underscore and take first part
        parts = name_str.split('_')
        sub_id = parts[0]
        return sub_id
    
    return None

def clean_dataframe(df, df_name="DataFrame"):
    """
    Remove columns that:
    1. Are completely empty (all NaN)
    2. Have identical values across all rows
    3. Have only one unique non-null value
    """
    print(f"\n  Cleaning {df_name}:")
    print(f"    Initial shape: {df.shape}")
    
    # Store original columns
    original_cols = df.columns.tolist()
    cols_to_drop = []
    
    for col in df.columns:
        # Skip the 'name' column (subject ID)
        if col.lower() in ['name', 'subject_id', 'participant_id']:
            continue
        
        # Check if column is completely empty
        if df[col].isna().all():
            cols_to_drop.append(col)
            print(f"    ✗ Removing '{col}': All values are NaN")
            continue
        
        # Get non-null values
        non_null_values = df[col].dropna()
        
        # Check if empty after dropping NaN
        if len(non_null_values) == 0:
            cols_to_drop.append(col)
            print(f"    ✗ Removing '{col}': No non-null values")
            continue
        
        # Check if all non-null values are identical
        unique_values = non_null_values.nunique()
        
        if unique_values == 1:
            constant_value = non_null_values.iloc[0]
            cols_to_drop.append(col)
            print(f"    ✗ Removing '{col}': All values identical (value={constant_value})")
            continue
    
    # Drop identified columns
    df_cleaned = df.drop(columns=cols_to_drop)
    
    print(f"    Final shape: {df_cleaned.shape}")
    print(f"    Removed {len(cols_to_drop)} columns")
    
    return df_cleaned, cols_to_drop

# Dictionary to store processed dataframes
processed_dfs = {}
all_subjects_found = set()

for csv_file in csv_files:
    print(f"\n📄 Processing: {csv_file.name}")
    
    # Load CSV
    df_smri = pd.read_csv(csv_file)
    print(f"  Loaded shape: {df_smri.shape}")
    
    # Check for 'name' column or similar
    name_columns = [col for col in df_smri.columns if 'name' in col.lower() or 'subject' in col.lower()]
    
    if not name_columns:
        print(f"  ⚠️ WARNING: No 'name' or 'subject' column found. Columns: {df_smri.columns.tolist()}")
        continue
    
    name_col = name_columns[0]
    print(f"  Using column '{name_col}' for subject IDs")
    
    # Extract subject IDs
    df_smri['subject_id'] = df_smri[name_col].apply(extract_subject_id)
    
    # Remove rows where subject_id couldn't be extracted
    df_smri = df_smri.dropna(subset=['subject_id'])
    
    print(f"  Found {len(df_smri)} subjects with valid IDs")
    
    # Filter for common subjects only
    df_smri_common = df_smri[df_smri['subject_id'].isin(common_subjects)].copy()
    
    print(f"  ✓ Subjects in common (EEG+fMRI): {len(df_smri_common)}")
    all_subjects_found.update(df_smri_common['subject_id'].tolist())
    
    # Clean the dataframe (remove empty/identical columns)
    df_smri_cleaned, dropped_cols = clean_dataframe(df_smri_common, csv_file.stem)
    
    # Merge with risk group information
    df_smri_final = df_smri_cleaned.merge(
        df_participants[['participant_id', 'Risk_Group', 'APOE_haplotype', 'PICALM_rs3851179']],
        left_on='subject_id',
        right_on='participant_id',
        how='left'
    )
    
    # Add group mapping (0, 1, 2)
    group_mapping = {'N': 0, 'A+P-': 1, 'A+P+': 2}
    df_smri_final['Group'] = df_smri_final['Risk_Group'].map(group_mapping)
    
    # Store processed dataframe
    processed_dfs[csv_file.stem] = df_smri_final
    
    print(f"  ✓ Final shape with risk groups: {df_smri_final.shape}")
    print(f"  ✓ Risk group distribution:")
    print(df_smri_final['Risk_Group'].value_counts().to_string(header=False).replace('\n', '\n    '))

# 4. SAVE PROCESSED FILES
print("\n" + "="*80)
print("STEP 4: SAVING PROCESSED sMRI DATA")
print("="*80)

output_dir = Path("sMRI_processed")
output_dir.mkdir(exist_ok=True)

for file_name, df in processed_dfs.items():
    output_file = output_dir / f"{file_name}_common_with_risk_groups.csv"
    df.to_csv(output_file, index=False)
    print(f"✓ Saved: {output_file}")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {df.shape[1]}")

# 5. CREATE COMBINED sMRI DATASET (OPTIONAL)
print("\n" + "="*80)
print("STEP 5: CREATING COMBINED sMRI DATASET")
print("="*80)

if len(processed_dfs) > 1:
    # Start with first dataframe
    first_key = list(processed_dfs.keys())[0]
    combined_df = processed_dfs[first_key][['subject_id', 'Risk_Group', 'Group']].copy()
    
    # Merge all dataframes
    for file_name, df in processed_dfs.items():
        # Get only the sMRI feature columns (exclude metadata)
        feature_cols = [col for col in df.columns if col not in 
                       ['subject_id', 'participant_id', 'Risk_Group', 'Group', 
                        'APOE_haplotype', 'PICALM_rs3851179', 'name']]
        
        # Add prefix to avoid column name conflicts
        df_features = df[['subject_id'] + feature_cols].copy()
        df_features.columns = ['subject_id'] + [f"{file_name}_{col}" for col in feature_cols]
        
        # Merge
        combined_df = combined_df.merge(df_features, on='subject_id', how='outer')
    
    # Save combined dataset
    combined_output = output_dir / "sMRI_all_features_combined.csv"
    combined_df.to_csv(combined_output, index=False)
    
    print(f"✓ Created combined dataset: {combined_output}")
    print(f"  Shape: {combined_df.shape}")
    print(f"  Total subjects: {len(combined_df)}")
    print(f"  Total features: {combined_df.shape[1] - 3}")  # Excluding subject_id, Risk_Group, Group

# 6. SUMMARY STATISTICS
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

print(f"\n📊 Overall Statistics:")
print(f"  Total unique subjects found in sMRI data: {len(all_subjects_found)}")
print(f"  Common subjects (EEG + fMRI): {len(common_subjects)}")
print(f"  sMRI subjects in common pool: {len(all_subjects_found.intersection(common_subjects))}")

print(f"\n📊 Risk Group Distribution (across all sMRI files):")
for file_name, df in processed_dfs.items():
    print(f"\n  {file_name}:")
    print(f"    N (Normal): {(df['Risk_Group'] == 'N').sum()}")
    print(f"    A+P- (APOE+, PICALM-): {(df['Risk_Group'] == 'A+P-').sum()}")
    print(f"    A+P+ (APOE+, PICALM+): {(df['Risk_Group'] == 'A+P+').sum()}")

print(f"\n📊 Files Saved:")
print(f"  Output directory: {output_dir}")
print(f"  Individual processed files: {len(processed_dfs)}")
if len(processed_dfs) > 1:
    print(f"  Combined file: sMRI_all_features_combined.csv")

# 7. DATA QUALITY REPORT
print("\n" + "="*80)
print("DATA QUALITY REPORT")
print("="*80)

for file_name, df in processed_dfs.items():
    print(f"\n📋 {file_name}:")
    
    # Check for missing values
    missing_summary = df.isnull().sum()
    cols_with_missing = missing_summary[missing_summary > 0]
    
    if len(cols_with_missing) > 0:
        print(f"  Columns with missing values:")
        for col, count in cols_with_missing.items():
            pct = (count / len(df)) * 100
            print(f"    • {col}: {count} ({pct:.1f}%)")
    else:
        print(f"  ✓ No missing values")
    
    # Feature statistics
    feature_cols = [col for col in df.columns if col not in 
                   ['subject_id', 'participant_id', 'Risk_Group', 'Group', 
                    'APOE_haplotype', 'PICALM_rs3851179', 'name']]
    
    print(f"  Total features: {len(feature_cols)}")
    print(f"  Numeric features: {df[feature_cols].select_dtypes(include=[np.number]).shape[1]}")
    print(f"  Non-numeric features: {df[feature_cols].select_dtypes(exclude=[np.number]).shape[1]}")

print("\n" + "="*80)
print("✅ sMRI DATA PROCESSING COMPLETE")
print("="*80)

In [ ]:
import nibabel as nib
from nilearn import datasets
import templateflow.api as tf

atlas_paths = tf.get(
    template='MNI152NLin2009cAsym', atlas='Schaefer2018',
    desc='100Parcels7Networks', extension='.nii.gz'
)

atlas_path = atlas_paths[0] if isinstance(atlas_paths, list) else atlas_paths
atlas_img = nib.load(atlas_path)
atlas_data = atlas_img.get_fdata()

# atlas names
labels = datasets.fetch_atlas_schaefer_2018(n_rois=100, yeo_networks=7)
atlas_labels = labels['labels']
print(f"Atlas labels: {atlas_labels}")

In [ ]:
import nibabel as nib
from nilearn import datasets, plotting
from nilearn.plotting import plot_img_on_surf
import templateflow.api as tf
import numpy as np
import matplotlib.pyplot as plt

# Load Schaefer atlas
atlas_paths = tf.get(
    template='MNI152NLin2009cAsym', atlas='Schaefer2018',
    desc='100Parcels7Networks', extension='.nii.gz'
)

atlas_path = atlas_paths[0] if isinstance(atlas_paths, list) else atlas_paths
atlas_img = nib.load(atlas_path)
atlas_data = atlas_img.get_fdata()

# Get atlas labels
labels = datasets.fetch_atlas_schaefer_2018(n_rois=100, yeo_networks=7)
atlas_labels = [label.decode('utf-8') for label in labels['labels']]

# Extract network names from labels
network_names = []
for label in atlas_labels:
    parts = label.split('_')
    network = parts[2]
    network_names.append(network)

# Get unique networks
unique_networks = sorted(set(network_names))
print(f"Found {len(unique_networks)} networks:")
for i, net in enumerate(unique_networks):
    print(f"  {i+1}. {net}")

# ====================================================================
# CONFIGURATION: SELECT WHICH NETWORKS TO HIGHLIGHT
# ====================================================================
networks_to_highlight = ['Default']  # Change this to your desired networks
# Example: ['Vis', 'Default'] for multiple networks
# networks_to_highlight = unique_networks  # Highlight all

print(f"\n✓ Highlighting networks: {networks_to_highlight}")
# ====================================================================

# Define colors for highlighted networks
network_colors = {
    'Vis': "#EAAAF7",        # Pastel Purple (Visual)
    'SomMot': "#9CA2F0",     # Pastel Blue (Somatomotor)
    'DorsAttn': "#F09C9C",   # Pastel Green (Dorsal Attention)
    'SalVentAttn': "#F0D55C", # Pastel Magenta (Salience/Ventral Attention)
    'Limbic': "#75AF6AFF",     # Very Light green (Limbic)
    'Cont': "#9CDCF0",       # Pastel Orange (Control)
    'Default': "#BE245F"     # Pastel Red (Default Mode)
}

# Create network-to-integer mapping (only for highlighted networks)
network_to_int = {}
highlight_counter = 1
for net in unique_networks:
    if net in networks_to_highlight:
        network_to_int[net] = highlight_counter
        highlight_counter += 1

# Create a new volume with NaN for non-highlighted regions (KEY CHANGE!)
network_atlas_data = np.full_like(atlas_data, np.nan, dtype=float)

for roi_idx, label in enumerate(atlas_labels):
    roi_value = roi_idx + 1
    network_name = network_names[roi_idx]
    
    # Only color the highlighted networks
    if network_name in networks_to_highlight:
        network_value = network_to_int[network_name]
        network_atlas_data[atlas_data == roi_value] = network_value

# Create a new NIfTI image
network_atlas_img = nib.Nifti1Image(network_atlas_data, atlas_img.affine, atlas_img.header)

# Create colormap (no need for 'white' anymore!)
colors_list = [network_colors[net] for net in networks_to_highlight]

from matplotlib.colors import ListedColormap
cmap = ListedColormap(colors_list)

# Plot on surface
fig = plt.figure(figsize=(12, 8))

plot_img_on_surf(
    network_atlas_img,
    views=["lateral", "medial"],
    hemispheres=["left", "right"],
    bg_on_data=True,
    cmap=cmap,
    vmin=0.5,
    vmax=len(networks_to_highlight) + 0.5,
    colorbar=False,

)



In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Network colors provided
network_colors = {
    'Vis': "#EAAAF7",        # Pastel Purple (Visual)
    'SomMot': "#9CA2F0",     # Pastel Blue (Somatomotor)
    'DorsAttn': "#F09C9C",   # Pastel Green (Dorsal Attention)
    'SalVentAttn': "#F0D55C", # Pastel Magenta (Salience/Ventral Attention)
    'Limbic': "#75AF6AFF",     # Very Light green (Limbic)
    'Cont': "#9CDCF0",       # Pastel Orange (Control)
    'Default': "#BE245F"     # Pastel Red (Default Mode)
}

# Network names (with more descriptive labels)
network_names = {
    'Vis': "Visual",
    'SomMot': "Somatomotor",
    'DorsAttn': "Dorsal Attention",
    'SalVentAttn': "Salience/Ventral Attention",
    'Cont': "Control",
    'Limbic': "Limbic",
    'Default': "Default Mode"
}

# Create figure with clean white background
plt.figure(figsize=(10, 3))
plt.rcParams['font.family'] = 'Arial'

# Create color patches
patches = []
for i, (key, color) in enumerate(network_colors.items()):
    patch = mpatches.Patch(color=color, label=network_names[key])
    patches.append(patch)

# Create legend
plt.legend(handles=patches, loc='center', ncol=4, frameon=False, fontsize=14)
plt.axis('off')  # Hide axes
plt.tight_layout()

# Save figure (uncomment to save to file)
# plt.savefig('network_colorbar.png', dpi=300, bbox_inches='tight', transparent=True)

plt.show()

In [ ]:
import nibabel as nib
from nilearn import plotting
from nilearn.plotting import plot_img_on_surf, view_img_on_surf
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap
import os
from pathlib import Path
import colorsys

# Path to the HCP-MMP1 atlas
atlas_path = "sMRI_atlas/HCPMMP1_on_MNI152_ICBM2009a_nlin_hd.nii.gz"

# Check if the file exists
if not os.path.exists(atlas_path):
    raise FileNotFoundError(f"Atlas file not found: {atlas_path}")

# Load the atlas
atlas_img = nib.load(atlas_path)
atlas_data = atlas_img.get_fdata()

# Get unique region IDs (excluding 0 which is background)
unique_regions = np.unique(atlas_data)
unique_regions = unique_regions[unique_regions != 0]  

n_regions = len(unique_regions)
print(f"Found {n_regions} regions in the HCP-MMP1 atlas")

# Create a new volume with integers from 1 to n_regions
# Each region gets a unique integer value for coloring
region_atlas_data = np.zeros_like(atlas_data)
for i, region_id in enumerate(unique_regions):
    region_atlas_data[atlas_data == region_id] = i + 1

# Create a new NIfTI image
region_atlas_img = nib.Nifti1Image(region_atlas_data, atlas_img.affine)

# ================================================================
# OPTION 1: VIBRANT COLORMAP - More saturated and varied colors
# ================================================================
def generate_vibrant_colors(n):
    """Generate vibrant colors with high contrast"""
    colors = []
    for i in range(n):
        # Use full color wheel with varied saturation and brightness
        hue = i / n
        # Higher saturation for vibrant colors (0.6-1.0 range)
        saturation = 0.6 + 0.4 * ((i % 5) / 4)
        # Varied brightness (0.7-1.0 range)
        value = 0.7 + 0.3 * ((i % 7) / 6)
        
        # Convert HSV to RGB
        r, g, b = colorsys.hsv_to_rgb(hue, saturation, value)
        colors.append((r, g, b))
    
    return colors

# ================================================================
# OPTION 2: PASTEL COLORMAP - Softer colors with good contrast
# ================================================================
def generate_pastel_colors(n):
    """Generate soft pastel colors"""
    colors = []
    for i in range(n):
        # Use HSV color space
        hue = i / n
        # Low saturation for pastel effect
        saturation = 0.3 + 0.2 * ((i % 3) / 2)
        # High value for brightness
        value = 0.9 + 0.1 * ((i % 4) / 3)
        
        # Convert HSV to RGB
        r, g, b = colorsys.hsv_to_rgb(hue, saturation, value)
        colors.append((r, g, b))
    
    return colors

# ================================================================
# OPTION 3: HIGH CONTRAST COLORMAP - Maximum distinction
# ================================================================
def generate_high_contrast_colors(n):
    """Generate highly contrasting colors"""
    colors = []
    
    # Use golden ratio for maximum color separation
    golden_ratio = 0.618033988749895
    h = 0
    
    for i in range(n):
        # Use golden ratio to get good distribution
        h = (h + golden_ratio) % 1.0
        
        # Alternate between high saturation and value
        if i % 3 == 0:
            s, v = 0.99, 0.99  # Bright and saturated
        elif i % 3 == 1:
            s, v = 0.7, 0.99   # Slightly less saturated but bright
        else:
            s, v = 0.99, 0.8   # Saturated but slightly darker
        
        r, g, b = colorsys.hsv_to_rgb(h, s, v)
        colors.append((r, g, b))
    
    return colors

# ================================================================
# OPTION 4: BUILT-IN COLORMAPS (wrapped to provide more colors)
# ================================================================
def extend_colormap(cmap_name, n):
    """Create extended colormap from built-in cmap to accommodate many regions"""
    base_cmap = plt.cm.get_cmap(cmap_name)
    # Take samples from the colormap and repeat as needed
    colors = []
    repeats = np.ceil(n / base_cmap.N).astype(int)
    
    for i in range(n):
        # Get color from base colormap with slight variations each time it wraps
        base_idx = i % base_cmap.N
        repeat_num = i // base_cmap.N
        
        # Get base color
        base_color = np.array(base_cmap(base_idx)[:3])
        
        # Adjust color slightly based on repeat number to create variation
        if repeat_num > 0:
            # Alternate between lighter and darker variations
            if repeat_num % 2 == 1:
                # Make lighter (but not too light)
                modified_color = base_color * 1.25
                modified_color = np.clip(modified_color, 0, 1)
            else:
                # Make darker (but not too dark)
                modified_color = base_color * 0.8
            colors.append(tuple(modified_color))
        else:
            colors.append(tuple(base_color))
    
    return colors

# ================================================================
# Create all the different colormaps
# ================================================================
vibrant_colors = generate_vibrant_colors(n_regions)
pastel_colors = generate_pastel_colors(n_regions)
high_contrast_colors = generate_high_contrast_colors(n_regions)
nipy_spectral_extended = extend_colormap('nipy_spectral', n_regions)
plasma_extended = extend_colormap('plasma', n_regions)

# Create ListedColormaps from each color set
vibrant_cmap = ListedColormap(vibrant_colors)
pastel_cmap = ListedColormap(pastel_colors)
high_contrast_cmap = ListedColormap(high_contrast_colors)
nipy_spectral_cmap = ListedColormap(nipy_spectral_extended)
plasma_cmap = ListedColormap(plasma_extended)

# ================================================================
# CHOOSE YOUR PREFERRED COLORMAP HERE:
# ================================================================
# Options: vibrant_cmap, pastel_cmap, high_contrast_cmap, 
#          nipy_spectral_cmap, plasma_cmap
current_cmap = high_contrast_cmap  # Change this to your preferred option

# Plot static visualization with enhanced contrast for contours
fig = plt.figure(figsize=(18, 12))
print(f"Plotting {n_regions} distinct regions with the selected colormap...")

# Plot with parameters that enhance region boundaries
plot_img_on_surf(
    region_atlas_img,
    views=["lateral", "medial"],
    hemispheres=["left", "right"],
    bg_on_data=False,      # Background not on data
    darkness=0.5,          # Increased darkness parameter for better edge contrast
    cmap="twilight",
    vmin=0.5,
    vmax=n_regions + 0.5,
    colorbar=False,
    threshold=0.1,         # Low threshold to ensure all regions are visible
)

plt.tight_layout()
plt.show()

# For interactive view
# view = view_img_on_surf(
#     region_atlas_img,
#     surf_mesh='fsaverage',
#     cmap=current_cmap,
#     vmin=0.5,
#     vmax=n_regions + 0.5,
#     threshold=0.1,
#     bg_on_data=False
# )
# view

In [ ]:
import nibabel as nib
from nilearn import datasets, plotting
from nilearn.plotting import plot_img_on_surf, view_img_on_surf
import templateflow.api as tf
import numpy as np
import matplotlib.pyplot as plt

# Load Schaefer atlas
atlas_paths = tf.get(
    template='MNI152NLin2009cAsym', atlas='Schaefer2018',
    desc='100Parcels7Networks', extension='.nii.gz'
)

atlas_path = atlas_paths[0] if isinstance(atlas_paths, list) else atlas_paths
atlas_img = nib.load(atlas_path)
atlas_data = atlas_img.get_fdata()

# Get atlas labels
labels = datasets.fetch_atlas_schaefer_2018(n_rois=100, yeo_networks=7)
atlas_labels = [label.decode('utf-8') for label in labels['labels']]

# Extract network names from labels
network_names = []
for label in atlas_labels:
    parts = label.split('_')
    network = parts[2]
    network_names.append(network)

# Get unique networks
unique_networks = sorted(set(network_names))
print(f"Found {len(unique_networks)} networks:")
for i, net in enumerate(unique_networks):
    print(f"  {i+1}. {net}")

# ====================================================================
# CONFIGURATION: DISPLAY ALL NETWORKS
# ====================================================================
networks_to_highlight = unique_networks  # Show ALL networks
print(f"\n✓ Highlighting networks: {networks_to_highlight}")
# ====================================================================

# Define colors for each network (matching the reference image)
network_colors = {
    'Vis': "#00995E",      # Purple (Visual)
    'SomMot': "#C9FFC4",   # Blue (Somatomotor) 
    'DorsAttn': "#C95FE9", # Green (Dorsal Attention)
    'SalVentAttn': "#E3C51A", # Red/Magenta (Salience/Ventral Attention)
    'Limbic': "#BA00C0",   # Cream/Yellow (Limbic)
    'Cont': "#1535E9",     # Orange (Control)
    'Default': "#74"#E6F05C" Red-Purple (Default Mode Network)
}

# Create network-to-integer mapping
network_to_int = {net: idx + 1 for idx, net in enumerate(unique_networks)}

# Create a new volume with NaN for background
network_atlas_data = np.full_like(atlas_data, np.nan, dtype=float)

for roi_idx, label in enumerate(atlas_labels):
    roi_value = roi_idx + 1
    network_name = network_names[roi_idx]
    network_value = network_to_int[network_name]
    network_atlas_data[atlas_data == roi_value] = network_value

# Create a new NIfTI image
network_atlas_img = nib.Nifti1Image(network_atlas_data, atlas_img.affine, atlas_img.header)

# Create colormap with all 7 network colors
colors_list = [network_colors[net] for net in unique_networks]

from matplotlib.colors import ListedColormap
cmap = ListedColormap(colors_list)

# Plot on surface with larger figure
fig = plt.figure(figsize=(16, 10))

view_img_on_surf(
    network_atlas_img,
    surf_mesh='fsaverage',
    # views=["top"],
    # hemispheres="left",
    bg_on_data=False,
    cmap=cmap,
    vmin=0.5,
    vmax=len(networks_to_highlight) + 0.5,
    colorbar=True,
)

# # Add legend
# from matplotlib.patches import Patch
# legend_elements = [
#     Patch(facecolor=network_colors[net], label=net) 
#     for net in unique_networks
# ]

# plt.legend(
#     handles=legend_elements,
#     loc='center',
#     bbox_to_anchor=(0.5, -0.05),
#     ncol=7,
#     frameon=False,
#     fontsize=12,
#     # title="Yeo 7 Networks"
# )

# plt.tight_layout()
# plt.show()

# from nilearn.plotting import view_img_on_surf

# view = view_img_on_surf(network_atlas_img)
# view


In [ ]:
import nibabel as nib
from nilearn import datasets, plotting
from nilearn.plotting import plot_img_on_surf, view_img_on_surf
import templateflow.api as tf
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# Load Schaefer atlas
atlas_paths = tf.get(
    template='MNI152NLin2009cAsym', atlas='Schaefer2018',
    desc='100Parcels7Networks', extension='.nii.gz'
)

atlas_path = atlas_paths[0] if isinstance(atlas_paths, list) else atlas_paths
atlas_img = nib.load(atlas_path)
atlas_data = atlas_img.get_fdata()

# Get atlas labels
labels = datasets.fetch_atlas_schaefer_2018(n_rois=100, yeo_networks=7)
atlas_labels = [label.decode('utf-8') for label in labels['labels']]

# Extract network names from labels
network_names = []
for label in atlas_labels:
    parts = label.split('_')
    network = parts[2]
    network_names.append(network)

# Get unique networks
unique_networks = sorted(set(network_names))
print(f"Found {len(unique_networks)} networks:")
for i, net in enumerate(unique_networks):
    print(f"  {i+1}. {net}")

# ====================================================================
# CONFIGURATION: DISPLAY ALL NETWORKS
# ====================================================================
networks_to_highlight = unique_networks  # Show ALL networks
print(f"\n✓ Highlighting networks: {networks_to_highlight}")
# ====================================================================

# CRITICAL CHANGE: Define colors with MORE CONTRAST AND BRIGHTNESS
network_colors = {
    'Vis': "#FF00FF",      # Magenta (Visual)
    'SomMot': "#0000FF",   # Bright Blue (Somatomotor)
    'DorsAttn': "#00FF00", # Bright Green (Dorsal Attention)
    'SalVentAttn': "#FF0000", # Pure Red (Salience/Ventral Attention)
    'Limbic': "#FFFF00",   # Bright Yellow (Limbic)
    'Cont': "#FF8000",     # Bright Orange (Control)
    'Default': "#00FFFF"   # Cyan (Default)
}

# NEW: Create a simple mapping - we'll manually set the order 
# to make sure we know exactly what color goes with what network
network_to_idx = {}
for i, net in enumerate(unique_networks):
    network_to_idx[net] = i + 1
    print(f"  Network {net} = value {i+1} = color {network_colors[net]}")

# Create a brand new volume filled with NaN
network_atlas_data = np.full_like(atlas_data, np.nan, dtype=float)

# Populate with integer values for each network
for roi_idx, label in enumerate(atlas_labels):
    roi_value = roi_idx + 1  # ROI values are 1-100
    network_name = network_names[roi_idx]
    network_idx = network_to_idx[network_name]  # Get integer value (1-7) for this network
    
    # Find voxels belonging to this ROI and assign network value
    roi_voxels = (atlas_data == roi_value)
    network_atlas_data[roi_voxels] = network_idx

# Create NIfTI image
network_atlas_img = nib.Nifti1Image(network_atlas_data, atlas_img.affine)

# Create colormap with maximum contrast
colors = [network_colors[net] for net in unique_networks]
cmap = ListedColormap(colors)

# Print final check
print("\nFinal check - unique values in data:")
unique_values = np.unique(network_atlas_data)
unique_values = unique_values[~np.isnan(unique_values)]  # Remove NaN
print(f"  Values in image: {unique_values}")
print(f"  Expected values: {list(range(1, len(unique_networks)+1))}")

# DISPLAY WITH PLOT_IMG_ON_SURF (STATIC PLOT) FIRST
# This helps diagnose issues before moving to interactive view
plt.figure(figsize=(12, 8))
plot_img_on_surf(
    network_atlas_img,
    views=["lateral", "medial"],
    hemispheres=["left", "right"], 
    cmap=cmap,
    vmin=0.5,  # Start below lowest value (1)
    vmax=len(unique_networks) + 0.5,  # End above highest value (7)
    threshold=0.1,  # Very low threshold to make sure everything shows
    bg_on_data=False,  # Background not shown over data for better contrast
)
plt.show()

# Now try interactive view with the same parameters
view = view_img_on_surf(
    network_atlas_img,
    surf_mesh='fsaverage',
    cmap=cmap,
    vmin=0.5, 
    vmax=len(unique_networks) + 0.5,
    threshold=0.1,  
    bg_on_data=False,
)

# Create a simple legend as a separate figure
plt.figure(figsize=(10, 2))
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=network_colors[net], label=net) 
    for net in unique_networks
]
plt.legend(handles=legend_elements, loc='center', ncol=7, frameon=False, fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()

# Return the interactive view
view

# FINAL


In [ ]:
from __future__ import annotations

import os
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from nilearn import datasets
from scipy.stats import (
    f_oneway,
    kruskal,
    levene,
    mannwhitneyu,
    shapiro,
    ttest_ind,
)
from sklearn.cluster import KMeans


# ============================================================================
# Configuration
# ============================================================================

FMRI_WINDOW_SEC = 60.0
FMRI_OVERLAP = 0.75
FMRI_STEP_SEC = FMRI_WINDOW_SEC * (1.0 - FMRI_OVERLAP)  # 15 seconds

K_STATES_LIST = list(range(4, 10))
SEED_LIST = [9, 42, 123, 2021, 2022]
P_VALUE_THRESHOLD = 0.05

PARTICIPANTS_TSV = "filtered_participants.tsv"

FMRI_DIR_CANDIDATES = [
    Path("data/fMRI_data/subject_windows_confounds_old_60sec"),
    Path("/data/s.dharia-ra/PEARL/MM-Mediation-Genes-AD/data/fMRI_data/subject_windows_confounds_old_60sec"),
]

EEG_DIR_CANDIDATES = [
    Path("/data/s.dharia-ra/PEARL/final_multi_modal/data/EEG_data/EEG_data_plv"),
    Path("data/EEG_data/EEG_data_plv"),
    Path("data/EEG_data/HFD_PSD_60sec_75overlap_6min"),
]

OUTPUT_ROOT = Path("fMRI_significant_clusters_60s_75overlap_multiseed")
COMBINED_UNIFIED_TOP_PAIRS_CSV = Path("fMRI_unified_top_fc_pairs_60s_75overlap_multiseed.csv")
SIGNIFICANT_CLUSTER_SUMMARY_CSV = Path("fMRI_significant_cluster_summary_60s_75overlap_multiseed.csv")

GROUP_LABEL_TO_NAME = {
    0: "N",
    1: "A+P-",
    2: "A+P+",
}

GROUP_LABEL_TO_SAFE_NAME = {
    0: "N",
    1: "AposPminus",
    2: "AposPpos",
}

SAFE_NAME_TO_READABLE = {
    "N": "N",
    "AposPminus": "A+P-",
    "AposPpos": "A+P+",
}


# ============================================================================
# Utilities
# ============================================================================

def resolve_existing_path(path_candidates: list[Path], label: str) -> Path:
    for path in path_candidates:
        if path.exists():
            return path
    tried = "\n".join(str(p) for p in path_candidates)
    raise FileNotFoundError(f"Could not find {label}. Tried:\n{tried}")


def extract_subject_id_from_filename(filename: str) -> str | None:
    match = re.search(r"sub-(\d+)", filename)
    if match:
        return match.group(1).zfill(2)

    match = re.match(r"(\d+)", filename)
    if match:
        return match.group(1).zfill(2)

    return None


def collect_subject_ids_from_directory(data_dir: Path, suffix: str = ".npz") -> set[str]:
    subject_ids: set[str] = set()
    for filename in os.listdir(data_dir):
        if not filename.endswith(suffix):
            continue
        subject_id = extract_subject_id_from_filename(filename)
        if subject_id is not None:
            subject_ids.add(subject_id)
    return subject_ids


def load_fmri_window_data(fmri_dir: Path) -> dict[str, np.ndarray]:
    X_list: list[np.ndarray] = []
    y_list: list[np.ndarray] = []
    sub_ids: list[str] = []

    label_stats: Counter[int] = Counter()

    for filename in sorted(os.listdir(fmri_dir)):
        if not filename.endswith("_windows.npz"):
            continue

        subject_id = extract_subject_id_from_filename(filename)
        if subject_id is None:
            continue

        file_path = fmri_dir / filename
        data = np.load(file_path, allow_pickle=True)

        subject_windows = np.concatenate((data["PA"], data["AP"]), axis=0)
        subject_label = int(np.asarray(data["label"]).item())

        X_list.append(subject_windows)
        y_list.append(np.full(subject_windows.shape[0], subject_label, dtype=int))
        sub_ids.extend([subject_id] * subject_windows.shape[0])
        label_stats[subject_label] += 1

    if not X_list:
        raise ValueError(f"No fMRI window files found in {fmri_dir}")

    X = np.concatenate(X_list, axis=0)
    y = np.concatenate(y_list, axis=0)
    sub_ids_array = np.asarray(sub_ids)

    print("=" * 80)
    print("Loaded fMRI window data")
    print("=" * 80)
    print(f"Directory: {fmri_dir}")
    print(f"X shape: {X.shape}")
    print(f"y shape: {y.shape}")
    print(f"Unique subjects: {len(np.unique(sub_ids_array))}")
    print(f"Subject label counts: {dict(label_stats)}")

    return {"X": X, "y": y, "sub_ids": sub_ids_array}


def compute_subject_dwell_times_fmri(
    X_data: np.ndarray,
    y_data: np.ndarray,
    sub_ids_data: np.ndarray,
    target_subjects: np.ndarray,
    kmeans_model: KMeans,
    k_states: int,
    step_sec: float,
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    subject_features: list[np.ndarray] = []
    subject_labels: list[int] = []
    processed_subjects: list[str] = []

    for subject in np.unique(target_subjects):
        subject_mask = sub_ids_data == subject
        if not np.any(subject_mask):
            continue

        subject_windows = X_data[subject_mask]
        subject_label = int(y_data[subject_mask][0])
        subject_states = kmeans_model.predict(subject_windows)
        subject_dwell_times = np.bincount(subject_states, minlength=k_states).astype(float) * step_sec

        subject_features.append(subject_dwell_times)
        subject_labels.append(subject_label)
        processed_subjects.append(subject)

    return np.asarray(subject_features), np.asarray(subject_labels), processed_subjects


def choose_statistical_test(groups: list[np.ndarray]) -> tuple[str, float, float]:
    if len(groups) < 2 or any(group.size == 0 for group in groups):
        return "None", 1.0, 0.0

    try:
        normality_ok = all(shapiro(group)[1] > 0.05 for group in groups if len(group) >= 3)
        equal_var_ok = levene(*groups)[1] > 0.05

        if normality_ok and equal_var_ok:
            if len(groups) == 2:
                stat, p_value = ttest_ind(*groups)
                return "t-test", float(p_value), float(stat)
            stat, p_value = f_oneway(*groups)
            return "ANOVA", float(p_value), float(stat)

        if len(groups) == 2:
            stat, p_value = mannwhitneyu(*groups)
            return "Mann-Whitney", float(p_value), float(stat)
        stat, p_value = kruskal(*groups)
        return "Kruskal-Wallis", float(p_value), float(stat)
    except Exception:
        return "None", 1.0, 0.0


def determine_dominant_group(
    subject_features: np.ndarray,
    subject_labels: np.ndarray,
    state: int,
) -> tuple[str, dict[int, float], str]:
    group_means: dict[int, float] = {}

    for label in np.unique(subject_labels):
        group_values = subject_features[subject_labels == label, state]
        group_means[int(label)] = float(np.mean(group_values))

    dominant_label = max(group_means, key=group_means.get)
    dominant_group_safe = GROUP_LABEL_TO_SAFE_NAME[dominant_label]

    means_str = "_".join(
        f"{GROUP_LABEL_TO_SAFE_NAME[label]}{group_means[label]:.1f}"
        for label in sorted(group_means)
    )

    return dominant_group_safe, group_means, means_str


def build_risk_group_dataframe(participants_path: Path, common_subjects: set[str]) -> pd.DataFrame:
    df = pd.read_csv(participants_path, sep="\t")

    conditions = [
        (~df["APOE_haplotype"].str.contains("e4", case=False))
        & (df["PICALM_rs3851179"] != "G/G"),
        (df["APOE_haplotype"].str.contains("e4", case=False))
        & (df["PICALM_rs3851179"] != "G/G"),
        (df["APOE_haplotype"].str.contains("e4", case=False))
        & (df["PICALM_rs3851179"] == "G/G"),
    ]
    choices = ["N", "A+P-", "A+P+"]
    df["Risk_Group"] = np.select(conditions, choices, default="Excluded")

    df.loc[
        (df["APOE_haplotype"].str.contains("e2", case=False))
        & (~df["APOE_haplotype"].str.contains("e4", case=False))
        & (df["Risk_Group"] == "N"),
        "Risk_Group",
    ] = "Excluded"

    study_df = df[df["Risk_Group"].isin(["N", "A+P-", "A+P+"])].copy()
    study_df["Has_EEG_fMRI"] = study_df["participant_id"].apply(
        lambda participant_id: participant_id.replace("sub-", "").zfill(2) in common_subjects
    )
    final_df = study_df[study_df["Has_EEG_fMRI"]].copy()
    final_df["Group"] = final_df["Risk_Group"].map({"N": 0, "A+P-": 1, "A+P+": 2})

    print("=" * 80)
    print("Participant filtering")
    print("=" * 80)
    print(f"participants.tsv path: {participants_path}")
    print(f"Final dataframe shape: {final_df.shape}")
    print("Class distribution:")
    print(final_df["Risk_Group"].value_counts())

    return final_df


def flat_to_matrix_coords_formula(flat_index: int, n_rois: int = 100) -> tuple[int, int]:
    i = n_rois - 2 - int(np.floor(np.sqrt(-8 * flat_index + 4 * n_rois * (n_rois - 1) - 7) / 2.0 - 0.5))
    j = flat_index + i + 1 - n_rois * (n_rois - 1) // 2 + (n_rois - i) * ((n_rois - i) - 1) // 2
    return i, j


def prettify_schaefer_label(label: str) -> str:
    return label.replace("7Networks_", "").replace("_", " ")


def aggregate_cluster_files(
    npz_files: list[Path],
    aggregation_within_cluster: str = "median",
) -> np.ndarray:
    if not npz_files:
        raise ValueError("No cluster files were provided for aggregation.")

    cluster_summaries: list[np.ndarray] = []
    for file_path in npz_files:
        data = np.load(file_path, allow_pickle=True)
        windows = data["windows"]

        if aggregation_within_cluster == "median":
            cluster_summary = np.median(windows, axis=0)
        elif aggregation_within_cluster == "mean":
            cluster_summary = np.mean(windows, axis=0)
        else:
            raise ValueError("aggregation_within_cluster must be 'median' or 'mean'")

        cluster_summaries.append(cluster_summary)

    return np.mean(np.asarray(cluster_summaries), axis=0)


def build_top_index_map(
    mean_vector: np.ndarray,
    top_n: int,
) -> tuple[list[int], dict[int, float]]:
    top_indices = np.argsort(mean_vector)[-top_n:][::-1]
    top_map = {int(idx): float(mean_vector[idx]) for idx in top_indices}
    return [int(idx) for idx in top_indices], top_map


def collect_cluster_files_from_dirs(cluster_dirs: list[Path]) -> list[Path]:
    cluster_files: list[Path] = []
    for cluster_dir in cluster_dirs:
        if cluster_dir.exists():
            cluster_files.extend(sorted(cluster_dir.glob("*.npz")))
    return cluster_files


def build_unified_direction_table(
    risk_cluster_dirs: list[Path],
    n_cluster_dirs: list[Path],
    output_csv: Path,
    top_n: int = 10,
    aggregation_within_cluster: str = "median",
) -> pd.DataFrame:
    risk_files = collect_cluster_files_from_dirs(risk_cluster_dirs)
    n_files = collect_cluster_files_from_dirs(n_cluster_dirs)

    if not risk_files:
        raise FileNotFoundError("No risk-higher cluster files found.")
    if not n_files:
        raise FileNotFoundError("No N-higher cluster files found.")

    risk_mean = aggregate_cluster_files(
        risk_files,
        aggregation_within_cluster=aggregation_within_cluster,
    )
    n_mean = aggregate_cluster_files(
        n_files,
        aggregation_within_cluster=aggregation_within_cluster,
    )

    risk_top_indices, risk_top_map = build_top_index_map(risk_mean, top_n=top_n)
    n_top_indices, n_top_map = build_top_index_map(n_mean, top_n=top_n)

    unified_indices = risk_top_indices + [idx for idx in n_top_indices if idx not in set(risk_top_indices)]

    labels = datasets.fetch_atlas_schaefer_2018(n_rois=100, yeo_networks=7)
    atlas_labels = [label.decode("utf-8") for label in labels["labels"]]

    rows: list[dict[str, object]] = []
    for rank, flat_idx in enumerate(unified_indices, start=1):
        roi_i, roi_j = flat_to_matrix_coords_formula(flat_idx, n_rois=100)
        region_1 = prettify_schaefer_label(atlas_labels[roi_i])
        region_2 = prettify_schaefer_label(atlas_labels[roi_j])

        rows.append(
            {
                "rank": rank,
                "roi_pair": f"({roi_i},{roi_j})",
                "region_names": f"{region_1} <-> {region_2}",
                "r_risk_higher_N_lt": risk_top_map.get(flat_idx, np.nan),
                "r_N_higher_N_gt": n_top_map.get(flat_idx, np.nan),
                "flat_index": flat_idx,
                "roi_i": roi_i,
                "roi_j": roi_j,
                "risk_present": flat_idx in risk_top_map,
                "n_present": flat_idx in n_top_map,
            }
        )

    unified_df = pd.DataFrame(rows)
    unified_df.to_csv(output_csv, index=False)

    printable_df = unified_df.copy()
    printable_df["r_risk_higher_N_lt"] = printable_df["r_risk_higher_N_lt"].map(
        lambda value: "—" if pd.isna(value) else f"{value:.4f}"
    )
    printable_df["r_N_higher_N_gt"] = printable_df["r_N_higher_N_gt"].map(
        lambda value: "—" if pd.isna(value) else f"{value:.4f}"
    )

    print("=" * 80)
    print("UNIFIED TOP FUNCTIONAL CONNECTIVITY TABLE")
    print("=" * 80)
    print(f"Risk-higher files used: {len(risk_files)}")
    print(f"N-higher files used: {len(n_files)}")
    print(f"Saved CSV: {output_csv}")
    print(
        printable_df[
            ["rank", "roi_pair", "region_names", "r_risk_higher_N_lt", "r_N_higher_N_gt"]
        ].to_string(index=False)
    )

    return unified_df


def summarize_top_connectivity_pairs(
    cluster_dir: Path,
    aggregation_within_cluster: str = "median",
    top_n: int = 10,
) -> pd.DataFrame:
    npz_files = sorted(cluster_dir.glob("*.npz"))
    if not npz_files:
        raise FileNotFoundError(f"No .npz files found in {cluster_dir}")

    mean_windows: list[np.ndarray] = []
    for file_path in npz_files:
        data = np.load(file_path, allow_pickle=True)
        windows = data["windows"]

        if aggregation_within_cluster == "median":
            cluster_summary = np.median(windows, axis=0)
        elif aggregation_within_cluster == "mean":
            cluster_summary = np.mean(windows, axis=0)
        else:
            raise ValueError("aggregation_within_cluster must be 'median' or 'mean'")

        mean_windows.append(cluster_summary)

    mean_windows_array = np.asarray(mean_windows)
    all_mean = np.mean(mean_windows_array, axis=0)

    labels = datasets.fetch_atlas_schaefer_2018(n_rois=100, yeo_networks=7)
    atlas_labels = [label.decode("utf-8") for label in labels["labels"]]

    top_indices = np.argsort(all_mean)[-top_n:][::-1]

    rows: list[dict[str, object]] = []
    for rank, flat_idx in enumerate(top_indices, start=1):
        roi_i, roi_j = flat_to_matrix_coords_formula(int(flat_idx), n_rois=100)
        rows.append(
            {
                "rank": rank,
                "flat_index": int(flat_idx),
                "roi_i": int(roi_i),
                "roi_j": int(roi_j),
                "region_1": atlas_labels[roi_i],
                "region_2": atlas_labels[roi_j],
                "mean_connectivity": float(all_mean[flat_idx]),
            }
        )

    results_df = pd.DataFrame(rows)

    print("=" * 80)
    print(f"Top {top_n} FC pairs from {cluster_dir}")
    print("=" * 80)
    print(f"Total cluster files: {len(npz_files)}")
    print(f"Flattened feature length: {len(all_mean)}")
    print(f"Expected upper-triangle length for 100 ROIs: {100 * 99 // 2}")
    print(results_df.to_string(index=False))

    return results_df


def make_grouped_output_dirs(base_dir: Path) -> dict[str, Path]:
    grouped_output_dirs = {
        "N": base_dir / "N_group_higher",
        "AposPminus": base_dir / "AposPminus_group_higher",
        "AposPpos": base_dir / "AposPpos_group_higher",
    }
    for path in grouped_output_dirs.values():
        path.mkdir(parents=True, exist_ok=True)
    return grouped_output_dirs


def create_seed_feature_dataframe(
    final_df_template: pd.DataFrame,
    participant_to_row_index: dict[str, int],
    significant_clusters: list[dict[str, object]],
    output_csv: Path,
) -> pd.DataFrame:
    seed_df = final_df_template.copy()

    for cluster in significant_clusters:
        k_value = cluster["K"]
        state = cluster["state"]
        dominant_group = cluster["dominant_group"]
        p_value = cluster["p_value"]

        col_name = f"fMRI_K{k_value}_state{state}_{dominant_group}"
        seed_df[col_name] = np.nan
        seed_df[f"{col_name}_pval"] = p_value
        seed_df[f"{col_name}_test"] = cluster["test"]

        for subject_idx, subject_id in enumerate(cluster["processed_subjects"]):
            participant_id = f"sub-{subject_id}"
            row_index = participant_to_row_index.get(participant_id)
            if row_index is None:
                continue
            seed_df.loc[row_index, col_name] = cluster["subject_features"][subject_idx]

    seed_df.to_csv(output_csv, index=False)
    return seed_df


def build_significant_cluster_summary(
    significant_clusters: list[dict[str, object]],
    output_csv: Path,
) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for cluster in significant_clusters:
        group_means = cluster["group_means"]
        rows.append(
            {
                "seed": cluster["seed"],
                "K": cluster["K"],
                "state": cluster["state"],
                "test": cluster["test"],
                "p_value": cluster["p_value"],
                "stat_value": cluster["stat_value"],
                "dominant_group": SAFE_NAME_TO_READABLE[cluster["dominant_group"]],
                "n_windows": cluster["n_windows"],
                "group_mean_N": group_means.get(0, np.nan),
                "group_mean_AposPminus": group_means.get(1, np.nan),
                "group_mean_AposPpos": group_means.get(2, np.nan),
                "means_str": cluster["means_str"],
            }
        )

    summary_df = pd.DataFrame(rows)
    summary_df.to_csv(output_csv, index=False)
    return summary_df


def run_single_seed_analysis(
    seed: int,
    fmri_data: dict[str, np.ndarray],
    final_df_template: pd.DataFrame,
    participant_to_row_index: dict[str, int],
) -> dict[str, object]:
    print()
    print("=" * 80)
    print(f"RUNNING SEED = {seed}")
    print("=" * 80)

    significant_clusters: list[dict[str, object]] = []
    unique_subjects = np.unique(fmri_data["sub_ids"])

    for k_states in K_STATES_LIST:
        print()
        print(f"Testing K_STATES = {k_states} | seed = {seed}")
        print("=" * 60)

        kmeans = KMeans(
            n_clusters=k_states,
            random_state=seed,
            n_init="auto",
            max_iter=300,
        )
        kmeans.fit(fmri_data["X"])

        all_cluster_predictions = kmeans.predict(fmri_data["X"])

        subject_features, subject_labels, processed_subjects = compute_subject_dwell_times_fmri(
            X_data=fmri_data["X"],
            y_data=fmri_data["y"],
            sub_ids_data=fmri_data["sub_ids"],
            target_subjects=unique_subjects,
            kmeans_model=kmeans,
            k_states=k_states,
            step_sec=FMRI_STEP_SEC,
        )

        for state in range(k_states):
            groups = [
                subject_features[subject_labels == label, state]
                for label in np.unique(subject_labels)
            ]
            test_name, p_value, stat_value = choose_statistical_test(groups)
            is_significant = p_value < P_VALUE_THRESHOLD
            marker = "***" if is_significant else ""

            print(f"  State {state}: {test_name}, p={p_value:.6f} {marker}")

            if not is_significant:
                continue

            dominant_group, group_means, means_str = determine_dominant_group(
                subject_features=subject_features,
                subject_labels=subject_labels,
                state=state,
            )

            cluster_mask = all_cluster_predictions == state
            cluster_windows = fmri_data["X"][cluster_mask]

            cluster_info = {
                "seed": seed,
                "K": k_states,
                "state": state,
                "p_value": p_value,
                "stat_value": stat_value,
                "test": test_name,
                "dominant_group": dominant_group,
                "group_means": group_means,
                "means_str": means_str,
                "windows": cluster_windows,
                "n_windows": int(len(cluster_windows)),
                "subject_features": subject_features[:, state].copy(),
                "processed_subjects": processed_subjects.copy(),
            }
            significant_clusters.append(cluster_info)

            readable_group = SAFE_NAME_TO_READABLE[dominant_group]
            print(f"    SIGNIFICANT: dominant group = {readable_group}")
            print(f"    Group means: {group_means}")
            print(f"    Windows: {len(cluster_windows)}")

    print()
    print("=" * 80)
    print(f"FOUND {len(significant_clusters)} SIGNIFICANT CLUSTERS FOR SEED {seed}")
    print("=" * 80)

    seed_output_root = OUTPUT_ROOT / f"seed_{seed}"
    grouped_output_dirs = make_grouped_output_dirs(seed_output_root)

    print()
    print("=" * 80)
    print(f"SAVING SIGNIFICANT CLUSTERS FOR SEED {seed}")
    print("=" * 80)

    for index, cluster in enumerate(significant_clusters, start=1):
        group_dir = grouped_output_dirs[cluster["dominant_group"]]
        filename = (
            f"cluster_seed{seed}_K{cluster['K']}_state{cluster['state']}_"
            f"{cluster['dominant_group']}_dominant_"
            f"p{cluster['p_value']:.6f}.npz"
        )
        file_path = group_dir / filename

        np.savez_compressed(
            file_path,
            windows=cluster["windows"],
            seed=cluster["seed"],
            K=cluster["K"],
            state=cluster["state"],
            p_value=cluster["p_value"],
            stat_value=cluster["stat_value"],
            test=cluster["test"],
            dominant_group=cluster["dominant_group"],
            group_means=cluster["group_means"],
            means_str=cluster["means_str"],
            n_windows=cluster["n_windows"],
            window_size_sec=FMRI_WINDOW_SEC,
            overlap=FMRI_OVERLAP,
            step_sec=FMRI_STEP_SEC,
        )

        print(f"{index:2d}. {file_path}")
        print(f"    Windows: {cluster['n_windows']}, p={cluster['p_value']:.6f}")

    seed_output_csv = seed_output_root / f"fMRI_all_significant_features_seed{seed}_60s_75overlap.csv"
    seed_df = create_seed_feature_dataframe(
        final_df_template=final_df_template,
        participant_to_row_index=participant_to_row_index,
        significant_clusters=significant_clusters,
        output_csv=seed_output_csv,
    )

    print()
    print("=" * 80)
    print(f"SUMMARY REPORT FOR SEED {seed}")
    print("=" * 80)
    print(f"Saved dataframe: {seed_output_csv}")
    print(f"Shape: {seed_df.shape}")
    print(f"Significant feature columns: {len(significant_clusters)}")

    dominant_counts = Counter(cluster["dominant_group"] for cluster in significant_clusters)
    print("\nClusters by dominant group:")
    for group_name, count in dominant_counts.items():
        print(f"  {SAFE_NAME_TO_READABLE[group_name]}: {count}")

    k_counts = Counter(cluster["K"] for cluster in significant_clusters)
    print("\nClusters by K value:")
    for k_value, count in sorted(k_counts.items()):
        print(f"  K={k_value}: {count}")

    if significant_clusters:
        p_values = [cluster["p_value"] for cluster in significant_clusters]
        print("\nP-value range:")
        print(f"  Min: {min(p_values):.6f}")
        print(f"  Max: {max(p_values):.6f}")
        print(f"  Mean: {np.mean(p_values):.6f}")

    for safe_group_name, group_dir in grouped_output_dirs.items():
        cluster_files = list(group_dir.glob("*.npz"))
        if not cluster_files:
            print()
            print(f"No significant clusters found for {SAFE_NAME_TO_READABLE[safe_group_name]}.")
            continue

        print()
        print(f"POST-HOC CONNECTIVITY SUMMARY FOR {SAFE_NAME_TO_READABLE[safe_group_name]} | seed = {seed}")
        summarize_top_connectivity_pairs(
            cluster_dir=group_dir,
            aggregation_within_cluster="median",
            top_n=10,
        )

    risk_higher_dirs = [
        grouped_output_dirs["AposPminus"],
        grouped_output_dirs["AposPpos"],
    ]
    n_higher_dirs = [grouped_output_dirs["N"]]
    seed_unified_csv = seed_output_root / f"fMRI_unified_top_fc_pairs_seed{seed}_60s_75overlap.csv"

    try:
        print()
        build_unified_direction_table(
            risk_cluster_dirs=risk_higher_dirs,
            n_cluster_dirs=n_higher_dirs,
            output_csv=seed_unified_csv,
            top_n=10,
            aggregation_within_cluster="median",
        )
    except FileNotFoundError as exc:
        print()
        print(f"Skipping unified direction table for seed {seed}: {exc}")

    return {
        "seed": seed,
        "significant_clusters": significant_clusters,
        "grouped_output_dirs": grouped_output_dirs,
        "seed_output_root": seed_output_root,
        "seed_output_csv": seed_output_csv,
    }


# ============================================================================
# Main workflow
# ============================================================================

def main() -> None:
    fmri_dir = resolve_existing_path(FMRI_DIR_CANDIDATES, "60s / 75% overlap fMRI directory")
    eeg_dir = resolve_existing_path(EEG_DIR_CANDIDATES, "EEG directory")
    participants_path = Path(PARTICIPANTS_TSV)

    if not participants_path.exists():
        raise FileNotFoundError(f"Participants file not found: {participants_path}")

    fmri_subjects = collect_subject_ids_from_directory(fmri_dir, suffix=".npz")
    eeg_subjects = collect_subject_ids_from_directory(eeg_dir, suffix=".npz")
    common_subjects = eeg_subjects.intersection(fmri_subjects)

    print("=" * 80)
    print("Data sources")
    print("=" * 80)
    print(f"fMRI dir: {fmri_dir}")
    print(f"EEG dir: {eeg_dir}")
    print(f"Number of subjects with both EEG and fMRI data: {len(common_subjects)}")

    fmri_data = load_fmri_window_data(fmri_dir)
    final_df = build_risk_group_dataframe(participants_path, common_subjects)
    participant_to_row_index = dict(zip(final_df["participant_id"], final_df.index))

    print()
    print("=" * 80)
    print("MULTI-SEED ANALYSIS")
    print("=" * 80)
    print(f"Seeds: {SEED_LIST}")
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    seed_results: list[dict[str, object]] = []
    all_significant_clusters: list[dict[str, object]] = []

    for seed in SEED_LIST:
        seed_result = run_single_seed_analysis(
            seed=seed,
            fmri_data=fmri_data,
            final_df_template=final_df,
            participant_to_row_index=participant_to_row_index,
        )
        seed_results.append(seed_result)
        all_significant_clusters.extend(seed_result["significant_clusters"])

    print()
    print("=" * 80)
    print("COMBINED MULTI-SEED SUMMARY")
    print("=" * 80)
    print(f"Seeds completed: {len(seed_results)}")
    print(f"Total significant clusters across all seeds: {len(all_significant_clusters)}")

    if all_significant_clusters:
        summary_df = build_significant_cluster_summary(
            significant_clusters=all_significant_clusters,
            output_csv=SIGNIFICANT_CLUSTER_SUMMARY_CSV,
        )
        print(f"Saved cluster summary CSV: {SIGNIFICANT_CLUSTER_SUMMARY_CSV}")
        print(f"Summary rows: {len(summary_df)}")

        seed_counts = Counter(cluster["seed"] for cluster in all_significant_clusters)
        print("\nClusters by seed:")
        for seed, count in sorted(seed_counts.items()):
            print(f"  seed={seed}: {count}")

        dominant_counts = Counter(cluster["dominant_group"] for cluster in all_significant_clusters)
        print("\nClusters by dominant group:")
        for group_name, count in dominant_counts.items():
            print(f"  {SAFE_NAME_TO_READABLE[group_name]}: {count}")

        k_counts = Counter(cluster["K"] for cluster in all_significant_clusters)
        print("\nClusters by K value:")
        for k_value, count in sorted(k_counts.items()):
            print(f"  K={k_value}: {count}")

        p_values = [cluster["p_value"] for cluster in all_significant_clusters]
        print("\nP-value range:")
        print(f"  Min: {min(p_values):.6f}")
        print(f"  Max: {max(p_values):.6f}")
        print(f"  Mean: {np.mean(p_values):.6f}")

        risk_higher_dirs: list[Path] = []
        n_higher_dirs: list[Path] = []
        for seed_result in seed_results:
            grouped_output_dirs = seed_result["grouped_output_dirs"]
            risk_higher_dirs.extend(
                [
                    grouped_output_dirs["AposPminus"],
                    grouped_output_dirs["AposPpos"],
                ]
            )
            n_higher_dirs.append(grouped_output_dirs["N"])

        print()
        build_unified_direction_table(
            risk_cluster_dirs=risk_higher_dirs,
            n_cluster_dirs=n_higher_dirs,
            output_csv=COMBINED_UNIFIED_TOP_PAIRS_CSV,
            top_n=10,
            aggregation_within_cluster="median",
        )
    else:
        print("No significant clusters were found across any seed.")

    print()
    print("=" * 80)
    print("COMPLETE")
    print("=" * 80)


if __name__ == "__main__":
    main()
